# 🧠 Brain Tumor MRI Classification — End-to-End Explainable Medical AI

![Python](https://img.shields.io/badge/Python-3.10%2B-blue?logo=python&logoColor=white)
![TensorFlow](https://img.shields.io/badge/TensorFlow-2.16%E2%80%932.21-FF6F00?logo=tensorflow&logoColor=white)
![Keras 3](https://img.shields.io/badge/Keras-3-D00000?logo=keras&logoColor=white)
![License: MIT](https://img.shields.io/badge/License-MIT-green.svg)

A complete, **single-notebook** project: data exploration, two trained models, full
evaluation, **Grad-CAM explainability**, advanced reliability analysis, and a clinical
decision-support simulation — all reproducible top-to-bottom.

**Sections**
1. **Phase 1 – Data Exploration** — dataset overview, class distribution, sample images, image-size & quality checks
2. **Phase 1 – Preprocessing** — datasets, resize/normalize, augmentation
3. **Phase 2 – CNN Baseline** — from-scratch benchmark
4. **Phase 2 – Transfer Learning** — EfficientNetB0 (freeze → fine-tune)
5. **Phase 2 – Evaluation** — confusion matrices, classification reports, model comparison
6. **Results and Discussion** — what the Phase 2 numbers mean
7. **Phase 3 – Explainable AI (Grad-CAM)** — heatmaps, interpretation, misclassification, confidence & class-level analysis
8. **Phase 4 – AI Application & Deployment Prep** — single-image prediction, patient cases, automated reports, performance dashboard, deployment discussion
9. **Phase 5 – Advanced Evaluation & Reliability** — error & confusion analysis, per-class metrics, confidence, calibration (ECE), explainability evaluation, model comparison, experiments
10. **Phase 6 – Clinical Decision Support System** — workflow, risk stratification, AI reports, human–AI collaboration, escalation policy, responsible-AI & deployment
11. **Final Results Summary & Executive Summary** — wrap-up for technical and non-technical readers

> ⏱️ **Runtime note:** training two CNNs on ~5,600 images is slow on CPU. Use a GPU if you
> can, or lower the `*_EPOCHS` values below. `EarlyStopping` already stops training once the
> validation loss stops improving. Phases 3–6 run in seconds to a couple of minutes once the
> models are trained.

> 🔁 **Reproducibility:** all RNGs are seeded (`SEED = 42`, set in the imports cell below). The
> released checkpoints in `models/` were trained on **TensorFlow 2.21 / Keras 3 (CPU)**; any
> TensorFlow `2.16–2.21` build runs the notebook, with metrics reproducing to within ≈0.5
> points. After training, both models are **saved to disk and immediately reloaded and
> re-evaluated** (Phase 2) so the numbers you read are the numbers in the released files. See
> [`docs/reproducibility.md`](../docs/reproducibility.md) for the full environment and assumptions.

> ⚕️ **Medical disclaimer:** research and educational use only. This is **not** a medical device
> and must not inform real diagnosis or treatment.


# Phase 1 – Data Exploration

**What:** understand the dataset before modelling — how many images, how they split across the
four tumour classes, what they look like, and whether the files are clean and consistently sized.

**Why:** EDA surfaces class imbalance, inconsistent image sizes, and corrupt files *before* they
silently corrupt training. It also tells us whether accuracy is a fair headline metric.

**Dataset & provenance:** the **Brain Tumor MRI Dataset** by Masoud Nickparvar
([Kaggle](https://www.kaggle.com/datasets/masoudnickparvar/brain-tumor-mri-dataset)), itself a
curated merge of the *figshare*, *SARTAJ*, and *Br35H* collections. It ships with an official
`Training/` and `Testing/` split and is **class-balanced** by design: each split holds an equal
number of images per class (a minority of files carry an `-aug-` prefix, i.e. the curator
augmented some classes to balance them). We honour the official split — `Testing/` is never seen
during training — so the held-out numbers are leakage-free.

**Expected outcome:** a balanced 4-class dataset (`glioma`, `meningioma`, `notumor`,
`pituitary`) of usable MRI images that we can confidently feed into a fixed-size pipeline.


In [1]:
# Imports
import os
import random
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import tensorflow as tf
from tensorflow.keras import layers, models

# ---- Reproducibility ----
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ---- Configuration (everything in one place) ----
IMAGE_SIZE  = 224
BATCH_SIZE  = 32
NUM_CLASSES = 4
VAL_SPLIT   = 0.2

# Training length (EarlyStopping may stop sooner). Lower these for a quick run.
CNN_EPOCHS = 15
TL_EPOCHS  = 10
FT_EPOCHS  = 5

# ---- Paths (this notebook lives in notebooks/, dataset is one level up) ----
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
TRAIN_DIR  = os.path.join(PROJECT_ROOT, "Dataset", "Training")
TEST_DIR   = os.path.join(PROJECT_ROOT, "Dataset", "Testing")
MODELS_DIR = os.path.join(PROJECT_ROOT, "models")
os.makedirs(MODELS_DIR, exist_ok=True)

print("TensorFlow:", tf.__version__)
print("Train dir :", TRAIN_DIR)
print("Test dir  :", TEST_DIR)

TensorFlow: 2.21.0
Train dir : <project_root>/Dataset/Training
Test dir  : <project_root>/Dataset/Testing


In [2]:
# Detect classes from sub-folders and count images per class (fast — no pixels loaded).
CLASS_NAMES = sorted(d for d in os.listdir(TRAIN_DIR)
                     if os.path.isdir(os.path.join(TRAIN_DIR, d)))

def count_images(directory):
    return {c: len(os.listdir(os.path.join(directory, c))) for c in CLASS_NAMES}

train_counts = count_images(TRAIN_DIR)
test_counts  = count_images(TEST_DIR)
print("Classes      :", CLASS_NAMES)
print("Train counts :", train_counts, "->", sum(train_counts.values()))
print("Test counts  :", test_counts, "->", sum(test_counts.values()))

Classes      : ['glioma', 'meningioma', 'notumor', 'pituitary']
Train counts : {'glioma': 1400, 'meningioma': 1400, 'notumor': 1400, 'pituitary': 1400} -> 5600
Test counts  : {'glioma': 400, 'meningioma': 400, 'notumor': 400, 'pituitary': 400} -> 1600


In [3]:
# Class distribution — training vs. testing (grouped bars).
# A balanced dataset means plain accuracy is a meaningful headline metric.
x = np.arange(NUM_CLASSES)
width = 0.4
fig, ax = plt.subplots(figsize=(9, 5))
b1 = ax.bar(x - width / 2, [train_counts[c] for c in CLASS_NAMES], width,
            label="Train", color="steelblue", edgecolor="black")
b2 = ax.bar(x + width / 2, [test_counts[c] for c in CLASS_NAMES], width,
            label="Test", color="indianred", edgecolor="black")
ax.bar_label(b1, padding=2, fontsize=9)
ax.bar_label(b2, padding=2, fontsize=9)
ax.set_xticks(x)
ax.set_xticklabels(CLASS_NAMES)
ax.set_title("Class Distribution — Training vs. Testing")
ax.set_xlabel("Class")
ax.set_ylabel("Number of images")
ax.legend()
plt.tight_layout()
plt.show()

<Figure size 900x500 with 1 Axes>

In [4]:
# Random sample images per class
fig, axes = plt.subplots(NUM_CLASSES, 4, figsize=(10, 2.6 * NUM_CLASSES))
for r, c in enumerate(CLASS_NAMES):
    class_dir = os.path.join(TRAIN_DIR, c)
    for j, fname in enumerate(random.sample(os.listdir(class_dir), 4)):
        img = Image.open(os.path.join(class_dir, fname)).convert("RGB")
        img = img.resize((IMAGE_SIZE, IMAGE_SIZE))
        axes[r, j].imshow(img); axes[r, j].axis("off")
        if j == 0:
            axes[r, j].set_title(c, loc="left", fontsize=11)
plt.suptitle("Sample MRI Images per Class", y=1.0)
plt.tight_layout(); plt.show()

<Figure size 1000x1040 with 16 Axes>

## Image Dimensions & Quality Checks

**What:** scan every training file's header to measure width/height and colour mode, and flag
any unreadable images.

**Why:** the models expect a fixed `224×224×3` input. If the raw images vary widely in size or
include grayscale/corrupt files, we need to know now — the data pipeline resizes and converts to
RGB, but only valid files survive that. Corrupt files would otherwise crash training mid-epoch.

**Expected outcome:** zero corrupt files and a clear view of the native resolution range we are
resizing down from.

In [5]:
# Image-size + colour-mode scan (reads file headers only via .size — fast, no full decode).
sizes, modes, corrupt = [], Counter(), []
for c in CLASS_NAMES:
    class_dir = os.path.join(TRAIN_DIR, c)
    for fname in os.listdir(class_dir):
        fpath = os.path.join(class_dir, fname)
        try:
            with Image.open(fpath) as im:
                sizes.append(im.size)          # (width, height)
                modes[im.mode] += 1
        except Exception:                      # truncated / non-image files
            corrupt.append(fpath)

widths  = np.array([s[0] for s in sizes])
heights = np.array([s[1] for s in sizes])
print(f"Scanned {len(sizes)} training images")
print(f"Corrupt/unreadable files : {len(corrupt)}")
print(f"Colour modes             : {dict(modes)}")
print(f"Width  range : {widths.min()}-{widths.max()} px (median {int(np.median(widths))})")
print(f"Height range : {heights.min()}-{heights.max()} px (median {int(np.median(heights))})")
print(f"Unique (w,h) sizes       : {len(set(sizes))}")

# Visualise the native resolutions we resize down from.
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.hist(widths, bins=30, color="steelblue", edgecolor="black", alpha=0.8, label="width")
ax1.hist(heights, bins=30, color="indianred", edgecolor="black", alpha=0.5, label="height")
ax1.axvline(IMAGE_SIZE, color="black", ls="--", label=f"target {IMAGE_SIZE}px")
ax1.set_title("Native image dimensions"); ax1.set_xlabel("pixels"); ax1.legend()
ax2.scatter(widths, heights, s=6, alpha=0.3, color="purple")
ax2.set_title("Width vs. height"); ax2.set_xlabel("width"); ax2.set_ylabel("height")
plt.tight_layout(); plt.show()

Scanned 5600 training images
Corrupt/unreadable files : 0
Colour modes             : {'L': 2358, 'RGB': 3238, 'RGBA': 3, 'P': 1}
Width  range : 150-1375 px (median 512)
Height range : 168-1446 px (median 512)
Unique (w,h) sizes       : 381


<Figure size 1200x400 with 2 Axes>

# Phase 1 – Preprocessing

**What:** turn the folder structure into efficient `tf.data` pipelines and standardise every
image to `224×224×3`.

**Why / best practice:**
* **Train / validation** come from `Training/` via a single seeded `validation_split`, so both
  models see the *same* split — a fair comparison with **no train/val leakage**.
* **Test** is the untouched `Testing/` folder, kept **unshuffled** so predictions stay aligned
  with their ground-truth labels.
* **Normalization lives inside each model** — a `Rescaling(1/255)` layer for the CNN, and
  EfficientNet's built-in normalization for the transfer model (it expects raw `[0,255]` pixels).
  Keeping it in-graph means the saved model is self-contained and can't be fed wrongly-scaled
  inputs at inference.

**Expected outcome:** three prefetched datasets (`train_ds`, `val_ds`, `test_ds`) ready for
training and evaluation.

In [6]:
# Build train / validation / test datasets from the directory structure.
train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR, validation_split=VAL_SPLIT, subset="training", seed=SEED,
    image_size=(IMAGE_SIZE, IMAGE_SIZE), batch_size=BATCH_SIZE, label_mode="int")

val_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR, validation_split=VAL_SPLIT, subset="validation", seed=SEED,
    image_size=(IMAGE_SIZE, IMAGE_SIZE), batch_size=BATCH_SIZE, label_mode="int")

test_ds = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR, image_size=(IMAGE_SIZE, IMAGE_SIZE), batch_size=BATCH_SIZE,
    label_mode="int", shuffle=False)   # keep order so labels align with predictions

CLASS_NAMES = train_ds.class_names      # read before prefetch wraps the dataset
print("Class names:", CLASS_NAMES)

# Prefetch so the CPU prepares the next batch while the model trains.
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds   = val_ds.prefetch(AUTOTUNE)
test_ds  = test_ds.prefetch(AUTOTUNE)

Found 5600 files belonging to 4 classes.
Using 4480 files for training.
Found 5600 files belonging to 4 classes.
Using 1120 files for validation.
Found 1600 files belonging to 4 classes.
Class names: ['glioma', 'meningioma', 'notumor', 'pituitary']


In [7]:
# Data augmentation block (rotation, flip, zoom). Active only during training.
# A small factory returns a fresh block so it can be embedded in multiple models safely.
def make_augmentation():
    return tf.keras.Sequential([
        layers.RandomFlip("horizontal", seed=SEED),
        layers.RandomRotation(0.05, seed=SEED),   # ~ +/- 18 degrees
        layers.RandomZoom(0.10, seed=SEED),
    ], name="data_augmentation")

# Preview augmentation on a single image.
demo_aug = make_augmentation()
images, _ = next(iter(train_ds))
plt.figure(figsize=(10, 5))
for i in range(8):
    aug = demo_aug(tf.expand_dims(images[0], 0), training=True)[0]
    plt.subplot(2, 4, i + 1)
    plt.imshow(tf.cast(aug, tf.uint8)); plt.axis("off")
plt.suptitle("Augmentation Preview (same image, random transforms)", y=1.0)
plt.tight_layout(); plt.show()

<Figure size 1000x500 with 8 Axes>

# Phase 2 – CNN Baseline

**What:** a transparent, from-scratch CNN — four `Conv → BatchNorm → ReLU → MaxPool` blocks,
global average pooling, and a dropout-regularised dense head over the 4 classes.

**Why:** it sets an honest benchmark to measure the transfer-learning model against.
**BatchNorm** stabilises and speeds up training, while **Dropout (0.3)** and **global average
pooling** combat overfitting on a modest dataset. The final conv layer is named `top_conv` so
Grad-CAM can target it in Phase 3.

**Expected outcome:** decent but unspectacular accuracy — the bar transfer learning must beat.

In [8]:
def build_cnn():
    """From-scratch CNN baseline.

    Four ``Conv -> BatchNorm -> ReLU -> MaxPool`` blocks (BatchNorm stabilises training and lets
    the model converge faster), global average pooling, and a dropout-regularised dense head.
    The final conv layer is named ``top_conv`` so Grad-CAM can target it in Phase 3.
    """
    model = models.Sequential([
        layers.Input((IMAGE_SIZE, IMAGE_SIZE, 3)),
        make_augmentation(),                 # augment during training only
        layers.Rescaling(1.0 / 255),         # normalize [0,255] -> [0,1]

        layers.Conv2D(32, 3, padding="same", use_bias=False),
        layers.BatchNormalization(), layers.Activation("relu"),
        layers.MaxPooling2D(),

        layers.Conv2D(64, 3, padding="same", use_bias=False),
        layers.BatchNormalization(), layers.Activation("relu"),
        layers.MaxPooling2D(),

        layers.Conv2D(128, 3, padding="same", use_bias=False),
        layers.BatchNormalization(), layers.Activation("relu"),
        layers.MaxPooling2D(),

        layers.Conv2D(128, 3, padding="same", use_bias=False, name="top_conv"),
        layers.BatchNormalization(), layers.Activation("relu"),
        layers.MaxPooling2D(),

        layers.GlobalAveragePooling2D(),
        layers.Dropout(0.3),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(NUM_CLASSES, activation="softmax"),
    ], name="cnn_baseline")

    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

cnn = build_cnn()
cnn.summary()

Model: "cnn_baseline"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ data_augmentation (Sequential)  │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling (Rescaling)           │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 224, 224, 32)   │           864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 224, 224, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 224, 224, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 112, 112, 64)   │        18,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 112, 112, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 112, 112, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 56, 56, 128)    │        73,728 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 56, 56, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_2 (Activation)       │ (None, 56, 56, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 28, 28, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ top_conv (Conv2D)               │ (None, 28, 28, 128)    │       147,456 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 28, 28, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_3 (Activation)       │ (None, 28, 28, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 14, 14, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼─────────────

 Total params: 258,916 (1011.39 KB)

 Trainable params: 258,212 (1008.64 KB)

 Non-trainable params: 704 (2.75 KB)

In [ ]:
# Train the CNN. Callbacks use a CONSISTENT monitor (val_loss) throughout:
#   * EarlyStopping     - stop when val_loss stops improving, RESTORE the best weights in memory.
#   * ReduceLROnPlateau - halve the LR when val_loss plateaus to settle the noisy training curve.
# We deliberately DO NOT use ModelCheckpoint here. EarlyStopping(restore_best_weights=True) already
# leaves the best-val_loss weights in memory; we then save THAT exact model to disk below, so the
# released checkpoint is guaranteed identical to the model we evaluate (see H1/H2 verification cell).
cnn_callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=5,
                                     restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                                         patience=2, min_lr=1e-5),
]
cnn_history = cnn.fit(train_ds, validation_data=val_ds,
                      epochs=CNN_EPOCHS, callbacks=cnn_callbacks)

# Persist the exact in-memory (best-val_loss) model.
cnn.save(os.path.join(MODELS_DIR, "cnn_model.keras"))
print("Saved:", os.path.join(MODELS_DIR, "cnn_model.keras"))


In [10]:
# Reusable helper: plot training vs. validation accuracy and loss.
def plot_history(history, title):
    h = history.history
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(h["accuracy"], label="train"); ax1.plot(h["val_accuracy"], label="val")
    ax1.set_title(f"{title} — Accuracy"); ax1.set_xlabel("Epoch"); ax1.legend(); ax1.grid(alpha=.3)
    ax2.plot(h["loss"], label="train"); ax2.plot(h["val_loss"], label="val")
    ax2.set_title(f"{title} — Loss"); ax2.set_xlabel("Epoch"); ax2.legend(); ax2.grid(alpha=.3)
    plt.tight_layout(); plt.show()

plot_history(cnn_history, "CNN")

<Figure size 1200x400 with 2 Axes>

# Phase 2 – Transfer Learning (EfficientNetB0)

We reuse an ImageNet-pretrained **EfficientNetB0** backbone in two stages:

1. **Stage 1 — feature extraction:** freeze the base, train only the new classification head at
   LR `1e-3`.
2. **Stage 2 — fine-tuning:** unfreeze the top ~20 layers and continue at a 100× smaller LR
   (`1e-5`) so the pretrained features adapt gently. **BatchNorm layers stay frozen** so their
   ImageNet running statistics aren't corrupted by our small batches.

**Why:** pretrained features transfer strongly to MRI, giving higher accuracy with far less data
and compute than training from scratch.

In [11]:
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input

def build_transfer():
    base = EfficientNetB0(include_top=False, weights="imagenet",
                          input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3))
    base.trainable = False                       # Stage 1: freeze the backbone

    inputs = layers.Input((IMAGE_SIZE, IMAGE_SIZE, 3))
    x = make_augmentation()(inputs)
    x = preprocess_input(x)                      # EfficientNet keras-port: expects [0,255];
                                                 # normalization is built into the backbone, so
                                                 # this call is effectively a pass-through.
    x = base(x, training=False)                  # nested 'efficientnetb0' (last conv: top_conv)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

    model = models.Model(inputs, outputs, name="efficientnet_transfer")
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model, base

effnet, effnet_base = build_transfer()
effnet.summary()

Model: "efficientnet_transfer"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_4 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ data_augmentation (Sequential)  │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 4)              │           516 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,214,055 (16.08 MB)

 Trainable params: 164,484 (642.52 KB)

 Non-trainable params: 4,049,571 (15.45 MB)

In [ ]:
# Stage 1: train the classification head (base frozen).
# Same consistent val_loss monitor as the CNN. No ModelCheckpoint: we save the final fine-tuned
# model explicitly after Stage 2 so the released file equals the evaluated model.
tl_ckpt = os.path.join(MODELS_DIR, "efficientnet_model.keras")
stage1_callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=5,
                                     restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                                         patience=2, min_lr=1e-6),
]
print("Stage 1: training head (base frozen)...")
effnet_history = effnet.fit(train_ds, validation_data=val_ds,
                            epochs=TL_EPOCHS, callbacks=stage1_callbacks)
plot_history(effnet_history, "EfficientNet (frozen base)")


In [ ]:
# Stage 2: unfreeze the top 20 base layers and fine-tune with a small learning rate.
effnet_base.trainable = True
for layer in effnet_base.layers[:-20]:
    layer.trainable = False
for layer in effnet_base.layers[-20:]:
    if isinstance(layer, layers.BatchNormalization):   # keep BN running stats frozen
        layer.trainable = False

effnet.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
               loss="sparse_categorical_crossentropy", metrics=["accuracy"])

# H1 FIX: use a FRESH EarlyStopping for Stage 2. Reusing the Stage-1 callbacks (or a shared
# save_best_only ModelCheckpoint pointed at the same file) would compare Stage-2 val_loss against
# Stage-1's best and could leave Stage-1 weights on disk — so the saved model would NOT be the
# fine-tuned model we evaluate. A fresh EarlyStopping restores the best STAGE-2 weights in memory.
stage2_callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=5,
                                     restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                                         patience=2, min_lr=1e-6),
]
print("Stage 2: fine-tuning top base layers...")
effnet_ft_history = effnet.fit(train_ds, validation_data=val_ds,
                               epochs=FT_EPOCHS, callbacks=stage2_callbacks)
plot_history(effnet_ft_history, "EfficientNet (fine-tuning)")

# H2 FIX: save the EXACT fine-tuned in-memory model. This single file is the released checkpoint.
effnet.save(tl_ckpt)
print("Saved:", tl_ckpt)


In [ ]:
# ── H1/H2/H3 verification: reload BOTH models from disk and use the on-disk copies downstream ──
# This is the guarantee behind every metric below: we discard the in-memory models and reload the
# exact files in models/. From here on, `cnn` and `effnet` ARE the released checkpoints, so what we
# evaluate is byte-for-byte what we ship. (Run order: this cell comes right after both models save.)
cnn    = tf.keras.models.load_model(os.path.join(MODELS_DIR, "cnn_model.keras"))
effnet = tf.keras.models.load_model(os.path.join(MODELS_DIR, "efficientnet_model.keras"))

for name, mdl in [("CNN", cnn), ("EfficientNetB0", effnet)]:
    loss, acc = mdl.evaluate(test_ds, verbose=0)
    print(f"Reloaded {name:<14} | test accuracy = {acc:.4f}")

print("Downstream evaluation now uses the reloaded (on-disk) models — evaluated == released.")


# Phase 2 – Evaluation

**What:** score both models on the held-out **test** set with the metrics that matter for a
4-class medical classifier — accuracy, per-class precision / recall / F1, and confusion matrices.

**Why:** accuracy alone hides class-specific failures. For tumour detection, **recall** per class
(did we catch every case?) is the metric to watch. Evaluating on the untouched test folder gives
an honest estimate of generalisation.

In [14]:
from sklearn.metrics import (accuracy_score, f1_score,
                             confusion_matrix, classification_report)

# Ground-truth labels for the (unshuffled) test set.
y_true = np.concatenate([y.numpy() for _, y in test_ds])

def predict_labels(model):
    return model.predict(test_ds).argmax(axis=1)

cnn_pred    = predict_labels(cnn)
effnet_pred = predict_labels(effnet)

print("CNN          test accuracy:", round(accuracy_score(y_true, cnn_pred), 4))
print("EfficientNet test accuracy:", round(accuracy_score(y_true, effnet_pred), 4))

50/50 ━━━━━━━━━━━━━━━━━━━━ 8s 153ms/step
50/50 ━━━━━━━━━━━━━━━━━━━━ 17s 309ms/step
CNN          test accuracy: 0.7119
EfficientNet test accuracy: 0.85


In [15]:
# Reusable confusion-matrix plot (matplotlib only).
def plot_confusion(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred, labels=range(NUM_CLASSES))
    plt.figure(figsize=(5.5, 5))
    plt.imshow(cm, cmap="Blues")
    plt.colorbar(fraction=0.046, pad=0.04)
    ticks = range(NUM_CLASSES)
    plt.xticks(ticks, CLASS_NAMES, rotation=45, ha="right")
    plt.yticks(ticks, CLASS_NAMES)
    thresh = cm.max() / 2
    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            plt.text(j, i, cm[i, j], ha="center",
                     color="white" if cm[i, j] > thresh else "black")
    plt.title(title); plt.ylabel("True label"); plt.xlabel("Predicted label")
    plt.tight_layout(); plt.show()

plot_confusion(y_true, cnn_pred, "CNN — Confusion Matrix")
plot_confusion(y_true, effnet_pred, "EfficientNet — Confusion Matrix")

<Figure size 550x500 with 2 Axes>

<Figure size 550x500 with 2 Axes>

In [16]:
print("================ CNN ================")
print(classification_report(y_true, cnn_pred, target_names=CLASS_NAMES,
                            labels=range(NUM_CLASSES)))
print("============ EfficientNet ============")
print(classification_report(y_true, effnet_pred, target_names=CLASS_NAMES,
                            labels=range(NUM_CLASSES)))

================ CNN ================
              precision    recall  f1-score   support

      glioma       0.66      0.71      0.68       400
  meningioma       0.62      0.45      0.52       400
     notumor       0.68      1.00      0.81       400
   pituitary       0.94      0.69      0.80       400

    accuracy                           0.71      1600
   macro avg       0.72      0.71      0.70      1600
weighted avg       0.72      0.71      0.70      1600

============ EfficientNet ============
              precision    recall  f1-score   support

      glioma       0.91      0.65      0.76       400
  meningioma       0.78      0.78      0.78       400
     notumor       0.90      0.99      0.95       400
   pituitary       0.82      0.99      0.90       400

    accuracy                           0.85      1600
   macro avg       0.85      0.85      0.84      1600
weighted avg       0.85      0.85      0.84      1600



## Model Comparison

A single side-by-side table of macro **Accuracy / Precision / Recall / F1** for both models.

In [17]:
from sklearn.metrics import precision_score, recall_score

# Macro-averaged metrics (every class weighted equally — fair on this balanced test set).
def score_row(name, y_pred):
    return {
        "Model":     name,
        "Accuracy":  accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, average="macro"),
        "Recall":    recall_score(y_true, y_pred, average="macro"),
        "F1 Score":  f1_score(y_true, y_pred, average="macro"),
    }

comparison = pd.DataFrame([
    score_row("CNN", cnn_pred),
    score_row("EfficientNetB0", effnet_pred),
]).round(4)
comparison

,Model,Accuracy,Precision,Recall,F1 Score
0,CNN,0.7119,0.7249,0.7119,0.7027
1,EfficientNetB0,0.8500,0.8549,0.8500,0.8441


## Sample Predictions & Misclassified Examples

Qualitative sanity check on the better model: a batch of predictions (green = correct,
red = wrong) followed by the actual misclassified cases across the full test set.

In [18]:
# Sample predictions from the better model (EfficientNet) on one test batch.
test_images, test_labels = next(iter(test_ds))
batch_pred = effnet.predict(test_images).argmax(axis=1)

plt.figure(figsize=(12, 7))
for i in range(12):
    plt.subplot(3, 4, i + 1)
    plt.imshow(tf.cast(test_images[i], tf.uint8)); plt.axis("off")
    true_c = CLASS_NAMES[int(test_labels[i])]
    pred_c = CLASS_NAMES[int(batch_pred[i])]
    plt.title(f"P: {pred_c}\nT: {true_c}",
              color="green" if true_c == pred_c else "red", fontsize=9)
plt.suptitle("EfficientNet — Sample Predictions (green = correct, red = wrong)", y=1.02)
plt.tight_layout(); plt.show()

1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step


<Figure size 1200x700 with 12 Axes>

In [19]:
# Misclassified examples across the full test set (EfficientNet).
mis_idx = np.where(effnet_pred != y_true)[0]
print(f"Misclassified: {len(mis_idx)} / {len(y_true)} "
      f"({100 * len(mis_idx) / len(y_true):.2f}%)")

# Materialize test images once (test set is small) to index the misclassified ones.
all_test_images = np.concatenate([x.numpy() for x, _ in test_ds])

n_show = min(8, len(mis_idx))
if n_show:
    plt.figure(figsize=(12, 6))
    for k, idx in enumerate(mis_idx[:n_show]):
        plt.subplot(2, 4, k + 1)
        plt.imshow(all_test_images[idx].astype("uint8")); plt.axis("off")
        plt.title(f"P: {CLASS_NAMES[effnet_pred[idx]]}\nT: {CLASS_NAMES[y_true[idx]]}",
                  color="red", fontsize=9)
    plt.suptitle("Misclassified Examples (EfficientNet)", y=1.02)
    plt.tight_layout(); plt.show()
else:
    print("No misclassified examples to show.")

Misclassified: 240 / 1600 (15.00%)


<Figure size 1200x600 with 8 Axes>

# Results and Discussion

**Headline:** the transfer-learning model (EfficientNetB0, **85% test accuracy**) clearly
outperforms the from-scratch CNN (**71%**) on every macro metric in the comparison table above —
the expected outcome, since ImageNet features transfer well to medical imaging and 4,480 training
images is modest for learning good convolutional filters from scratch.

**Where the errors concentrate**
* **`notumor`** is the easiest class (recall 0.99 for EfficientNet, 1.00 for the CNN) — "no
  tumour" MRIs are visually distinct from the three tumour types.
* **`glioma` → `meningioma` is the single largest confusion** for EfficientNet (81 cases, ~20% of
  all gliomas), and **glioma is its weakest class (recall 0.65)** — the two share appearance and
  location cues on a single 2-D slice.
* **The CNN's weak spot is different:** its lowest recall is on **`meningioma` (0.45**, vs 0.71 for
  glioma), so the from-scratch model struggles most to separate meningioma from the other types.
  Each model's lower scores trace to a specific class boundary, not a systemic failure.

**Why transfer learning wins here**
* Pretrained EfficientNet filters already encode edges, textures, and shapes; the model only has
  to learn a 4-class head plus light fine-tuning of the top blocks.
* Two-stage training (freeze → fine-tune at a 100× smaller LR) adapts those features without
  destroying them — and keeping BatchNorm frozen during fine-tuning avoids corrupting the
  pretrained running statistics.

**Limitations / next steps**
* Macro metrics weight all classes equally — appropriate here because the test set is balanced.
  On an imbalanced deployment set, watch per-class **recall** (missing a tumour is the costly
  error).
* No external validation set: results may be optimistic for scanners/populations unlike this
  dataset.
* **Phase 3 – Explainable AI:** Grad-CAM (implemented in Phase 3 below) confirms whether the
  models attend to the tumour region rather than scanner artefacts — critical before any
  clinical trust.

---

### Pipeline summary
This single notebook runs end-to-end: **EDA** (distribution, image-size & quality checks) →
**preprocessing** (leakage-free split, in-graph normalization, augmentation) → **two models**
(CNN baseline + EfficientNetB0 freeze→fine-tune, with EarlyStopping + ModelCheckpoint +
ReduceLROnPlateau) → **evaluation** (confusion matrices, classification reports, comparison
table) → **predictions** (sample + misclassified) → **Phase 3** (Grad-CAM explainability).

Best models are saved to `models/cnn_model.keras` and `models/efficientnet_model.keras`.

---

# Phase 3 – Explainable AI (Grad-CAM)

A medical classifier is only trustworthy if we can see *why* it decides what it decides. This
phase adds **Grad-CAM** (Gradient-weighted Class Activation Mapping) to our best Phase 2 model,
**EfficientNetB0**, turning a black-box classifier into an interpretable one.

**How Grad-CAM works (in one line):** it weighs the final convolutional feature maps by how
strongly each one influences the predicted class, producing a heatmap of the image regions that
drove the decision.

This section covers a reusable Grad-CAM implementation, per-example interpretation, correct- and
mis-classification analysis, model-confidence reliability, and per-class focus checks — closing
with a discussion of explainable AI in clinical practice and a final results summary.

## Grad-CAM Implementation

The next cell builds the toolkit once and reuses it everywhere below:

* **`locate_last_conv`** — traces one forward pass to find the deepest 4-D feature map. This
  works for plain CNNs *and* models wrapping a nested pretrained backbone, and raises a clear
  error for non-convolutional models.
* **`make_gradcam_heatmap`** — runs the model layer-by-layer, captures that feature map inside a
  `GradientTape`, and returns a ReLU-normalised, class-discriminative heatmap.
* **`colorize_and_overlay` / `explain_image`** — high-quality overlays, including inference on an
  arbitrary MRI file (single-image support).

> **Why not the textbook `Model(model.input, [conv.output, model.output])`?** Under **Keras 3** a
> nested pretrained backbone (EfficientNet) keeps its own internal graph, so slicing its
> `.output` raises *"Output … is not connected to inputs."* Walking the layers in a live forward
> pass avoids that entirely — and is architecture-agnostic.
>
> The model normalises pixels internally, so Grad-CAM is fed **raw `[0,255]`** images; augmentation
> layers are inactive at inference, so heatmaps are deterministic.

In [20]:
# ============================================================================
# Grad-CAM core  (Keras-3 robust)
# ============================================================================
# NOTE: the textbook recipe `tf.keras.Model(model.input, [conv_layer.output, model.output])`
# FAILS here with "Output ... is not connected to inputs". Under Keras 3 a *nested* pretrained
# backbone (EfficientNet) keeps its own internal graph, so its `.output` tensor is not wired to
# the outer model's input. Instead we run a short layer-by-layer forward pass and capture the
# last conv feature map directly from the live computation -- which is also architecture-
# agnostic (it works for the plain CNN too).
import matplotlib as mpl

best_model      = effnet                 # EfficientNetB0 was the stronger Phase 2 model
BEST_MODEL_NAME = "EfficientNetB0"
DISPLAY = {"glioma": "Glioma", "meningioma": "Meningioma",
           "notumor": "No Tumor", "pituitary": "Pituitary"}

def _callable_layers(model):
    """Top-level layers excluding the InputLayer (which is not callable)."""
    return [l for l in model.layers if not isinstance(l, tf.keras.layers.InputLayer)]

def locate_last_conv(model, sample):
    """Trace one forward pass and return (layers, conv_idx, conv_name) for the deepest
    layer that emits a 4-D (H, W, C) feature map.

    Works for plain CNNs *and* models that wrap a nested pretrained backbone, and raises a
    clear error for non-convolutional (unsupported) models."""
    layers = _callable_layers(model)
    x = tf.convert_to_tensor(sample, tf.float32)
    conv_idx = None
    for i, layer in enumerate(layers):
        x = layer(x, training=False)
        if len(x.shape) == 4:
            conv_idx = i
    if conv_idx is None:
        raise ValueError(
            f"Grad-CAM needs a convolutional feature map, but model {model.name!r} has no "
            f"4-D layer output. Grad-CAM supports CNN-based models only.")
    return layers, conv_idx, layers[conv_idx].name

# Resolve the target layer once (traced on a real test image) and reuse it everywhere.
GC_LAYERS, GC_CONV_IDX, GC_CONV_NAME = locate_last_conv(best_model, all_test_images[:1])
print(f"Grad-CAM target layer in {BEST_MODEL_NAME}: '{GC_CONV_NAME}'  "
      f"(last 4-D feature map before the classifier head)")

def make_gradcam_heatmap(img_batch, pred_index=None):
    """Grad-CAM heatmap for one raw-[0,255] image batch of shape (1, H, W, 3).
    Returns (heatmap[h, w] in [0, 1], predicted_index, probability_vector)."""
    with tf.GradientTape() as tape:
        x = tf.convert_to_tensor(img_batch, tf.float32)
        for layer in GC_LAYERS[:GC_CONV_IDX + 1]:      # input -> last conv feature map
            x = layer(x, training=False)
        conv_out = x
        tape.watch(conv_out)                           # watch the intermediate feature map
        for layer in GC_LAYERS[GC_CONV_IDX + 1:]:      # feature map -> class probabilities
            x = layer(x, training=False)
        preds = x
        if pred_index is None:
            pred_index = int(tf.argmax(preds[0]))
        class_channel = preds[:, pred_index]
    grads = tape.gradient(class_channel, conv_out)     # d(score) / d(feature map)
    if grads is None:
        raise RuntimeError("Grad-CAM gradient is None - target layer not connected to output.")
    pooled  = tf.reduce_mean(grads, axis=(0, 1, 2))    # per-channel importance weights
    heatmap = tf.squeeze(conv_out[0] @ pooled[..., None])              # weighted channel sum
    heatmap = tf.maximum(heatmap, 0) / (tf.reduce_max(heatmap) + 1e-8) # ReLU + normalise
    return heatmap.numpy(), pred_index, preds[0].numpy()

def colorize_and_overlay(img_uint8, heatmap, alpha=0.5, cmap="jet"):
    """Resize the heatmap to the scan, colour-map it, and alpha-blend.

    The blend is *intensity-weighted*: cold (low-activation) regions keep the original scan
    crisp while only the regions the model attends to are tinted. This reads far more clearly
    than a flat whole-image blend, which washes the entire scan in the colour map's cold end
    and hides anatomy -- the difference between a demo overlay and a publication-quality one."""
    h, w = img_uint8.shape[:2]
    hm = np.array(Image.fromarray(np.uint8(255 * heatmap)).resize((w, h), Image.BILINEAR)) / 255.0
    colored = (mpl.colormaps[cmap](hm)[..., :3] * 255).astype("uint8")
    weight  = (alpha * hm)[..., None]                                  # paint only where it looks
    overlay = np.clip(img_uint8 * (1 - weight) + colored * weight, 0, 255).astype("uint8")
    return colored, overlay

def load_image(path, size=IMAGE_SIZE):
    """Load an arbitrary MRI file as a raw-[0,255] float array (model-ready)."""
    return np.array(Image.open(path).convert("RGB").resize((size, size))).astype("float32")

# Cache test-set probabilities once (reused by every Phase 3 section).
effnet_probs = best_model.predict(test_ds, verbose=0)
pred_all = effnet_probs.argmax(axis=1)
conf_all = effnet_probs.max(axis=1)
assert np.array_equal(pred_all, effnet_pred), "Prediction mismatch vs. Phase 2"
print(f"Cached probabilities for {len(pred_all)} test images "
      f"(mean confidence {conf_all.mean():.1%}).")

Grad-CAM target layer in EfficientNetB0: 'efficientnetb0'  (last 4-D feature map before the classifier head)
Cached probabilities for 1600 test images (mean confidence 89.4%).


In [21]:
# ---- Grad-CAM visualisation + selection helpers (reused across all sections) ----
def _gradcam_for_index(idx):
    """Run Grad-CAM on test image `idx`; returns image, heatmap, overlay, pred, probs."""
    img = all_test_images[idx].astype("uint8")
    heatmap, pred_i, probs = make_gradcam_heatmap(all_test_images[idx][None].astype("float32"))
    colored, overlay = colorize_and_overlay(img, heatmap)
    return img, colored, overlay, pred_i, probs

def gradcam_panel(indices, suptitle):
    """One row per image: Original | Heatmap | Overlay (+ pred / true / confidence)."""
    indices = list(indices)
    fig, axes = plt.subplots(len(indices), 3, figsize=(9, 3 * len(indices)))
    axes = np.atleast_2d(axes)
    for r, idx in enumerate(indices):
        img, colored, overlay, pred_i, probs = _gradcam_for_index(idx)
        true_i, conf = int(y_true[idx]), probs[pred_i]
        ok = pred_i == true_i
        axes[r, 0].imshow(img);     axes[r, 0].set_title("Original MRI", fontsize=10)
        axes[r, 1].imshow(colored); axes[r, 1].set_title("Grad-CAM heatmap", fontsize=10)
        axes[r, 2].imshow(overlay)
        axes[r, 2].set_title(f"Pred: {DISPLAY[CLASS_NAMES[pred_i]]} ({conf:.0%})\n"
                             f"True: {DISPLAY[CLASS_NAMES[true_i]]}",
                             color="green" if ok else "red", fontsize=10)
        for a in axes[r]:
            a.axis("off")
    plt.suptitle(suptitle, y=1.0, fontsize=13)
    plt.tight_layout(); plt.show()

def explain_image(path):
    """Grad-CAM on an arbitrary MRI file (demonstrates single-image inference)."""
    arr = load_image(path)
    heatmap, pred_i, probs = make_gradcam_heatmap(arr[None])
    colored, overlay = colorize_and_overlay(arr.astype("uint8"), heatmap)
    fig, ax = plt.subplots(1, 3, figsize=(9, 3))
    for a, im, t in zip(ax, [arr.astype("uint8"), colored, overlay],
                        ["Original MRI", "Grad-CAM heatmap", "Overlay"]):
        a.imshow(im); a.set_title(t, fontsize=10); a.axis("off")
    plt.suptitle(f"Prediction: {DISPLAY[CLASS_NAMES[pred_i]]}  -  confidence {probs[pred_i]:.1%}",
                 y=1.02, fontsize=12)
    plt.tight_layout(); plt.show()
    return CLASS_NAMES[pred_i], float(probs[pred_i])

def pick(true_c=None, pred_c=None, correct=None, n=4):
    """Test-set indices filtered by class/correctness, sorted by confidence (desc)."""
    mask = np.ones(len(y_true), dtype=bool)
    if true_c is not None: mask &= (y_true == true_c)
    if pred_c is not None: mask &= (pred_all == pred_c)
    if correct is True:    mask &= (pred_all == y_true)
    if correct is False:   mask &= (pred_all != y_true)
    idx = np.where(mask)[0]
    return idx[np.argsort(-conf_all[idx])][:n]

def auto_explanation(pred_i, true_i, conf):
    """Plain-language rationale for a prediction, driven by class + confidence."""
    cls = DISPLAY[CLASS_NAMES[pred_i]]
    region = ("uniform tissue with no focal tumour region" if CLASS_NAMES[pred_i] == "notumor"
              else f"the highlighted brain region associated with characteristics of {cls}")
    if pred_i == true_i:
        if conf >= 0.85:
            return f"The model confidently predicted {cls} ({conf:.0%}), focusing primarily on {region}."
        return (f"The model predicted {cls} with moderate confidence ({conf:.0%}); "
                f"it attended to {region}, though less decisively.")
    return (f"The model misclassified a {DISPLAY[CLASS_NAMES[true_i]]} scan as {cls} ({conf:.0%}). "
            f"Its attention fell on regions it associated with {cls}, which explains the error.")

In [22]:
# Full Grad-CAM panels (Original | Heatmap | Overlay) on a few confident, correct predictions.
gradcam_panel(pick(correct=True, n=3), "Grad-CAM — Correctly Classified Examples")

# Single-image inference on a raw MRI file (works outside the test pipeline too).
glioma_dir  = os.path.join(TEST_DIR, "glioma")
sample_path = os.path.join(glioma_dir, sorted(os.listdir(glioma_dir))[0])
print("Single-image demo on:", os.path.relpath(sample_path, PROJECT_ROOT))
explain_image(sample_path);

<Figure size 900x900 with 9 Axes>

Single-image demo on: Dataset\Testing\glioma\Te-gl_1.jpg


<Figure size 900x300 with 3 Axes>

---

## Model Interpretation

For one confident, correct example per class we show the **original scan** and the **Grad-CAM
overlay**, then print the **actual** and **predicted** labels, the **confidence score**, and a
short plain-language explanation generated automatically from the prediction.

In [23]:
# One representative high-confidence example per class, each with a plain-language rationale.
for c in range(NUM_CLASSES):
    sel = pick(true_c=c, correct=True, n=1)
    if not len(sel):
        continue
    idx = sel[0]
    img, colored, overlay, pred_i, probs = _gradcam_for_index(idx)
    true_i, conf = int(y_true[idx]), probs[pred_i]

    fig, ax = plt.subplots(1, 2, figsize=(6.5, 3.2))
    ax[0].imshow(img);     ax[0].set_title("Original MRI", fontsize=10); ax[0].axis("off")
    ax[1].imshow(overlay); ax[1].set_title("Grad-CAM overlay", fontsize=10); ax[1].axis("off")
    plt.tight_layout(); plt.show()

    print(f"Actual     : {DISPLAY[CLASS_NAMES[true_i]]}")
    print(f"Predicted  : {DISPLAY[CLASS_NAMES[pred_i]]}")
    print(f"Confidence : {conf:.1%}")
    print("Explanation:", auto_explanation(pred_i, true_i, conf))
    print("-" * 84)

<Figure size 650x320 with 2 Axes>

Actual     : Glioma
Predicted  : Glioma
Confidence : 100.0%
Explanation: The model confidently predicted Glioma (100%), focusing primarily on the highlighted brain region associated with characteristics of Glioma.
------------------------------------------------------------------------------------


<Figure size 650x320 with 2 Axes>

Actual     : Meningioma
Predicted  : Meningioma
Confidence : 99.9%
Explanation: The model confidently predicted Meningioma (100%), focusing primarily on the highlighted brain region associated with characteristics of Meningioma.
------------------------------------------------------------------------------------


<Figure size 650x320 with 2 Axes>

Actual     : No Tumor
Predicted  : No Tumor
Confidence : 100.0%
Explanation: The model confidently predicted No Tumor (100%), focusing primarily on uniform tissue with no focal tumour region.
------------------------------------------------------------------------------------


<Figure size 650x320 with 2 Axes>

Actual     : Pituitary
Predicted  : Pituitary
Confidence : 100.0%
Explanation: The model confidently predicted Pituitary (100%), focusing primarily on the highlighted brain region associated with characteristics of Pituitary.
------------------------------------------------------------------------------------


---

## Correct Prediction Examples

Several correctly classified scans per class, shown as Grad-CAM overlays (rows = classes). These
are the model's most confident correct calls for each tumour type.

In [24]:
# Grid of correctly classified Grad-CAM overlays: rows = classes, columns = examples.
N_PER_CLASS = 3
fig, axes = plt.subplots(NUM_CLASSES, N_PER_CLASS, figsize=(3 * N_PER_CLASS, 3 * NUM_CLASSES))
for r, c in enumerate(range(NUM_CLASSES)):
    idxs = pick(true_c=c, correct=True, n=N_PER_CLASS)
    for j in range(N_PER_CLASS):
        ax = axes[r, j]
        if j < len(idxs):
            _, _, overlay, pred_i, probs = _gradcam_for_index(idxs[j])
            ax.imshow(overlay)
            ax.set_title(f"{DISPLAY[CLASS_NAMES[c]]} ({probs[pred_i]:.0%})", fontsize=10)
        ax.axis("off")
plt.suptitle("Correctly Classified — Grad-CAM Overlays by Class", y=1.0, fontsize=13)
plt.tight_layout(); plt.show()

<Figure size 900x1200 with 12 Axes>

**What the model focuses on, class by class.**

* **Glioma** — heat concentrates on the irregular, often infiltrative lesion mass. *Strength:* it
  reliably finds the abnormal region. *Weakness:* the diffuse margins overlap with meningioma,
  which is the main source of confusion (see the misclassification analysis below).
* **Meningioma** — attention sits on the well-circumscribed, extra-axial mass near the meninges.
  *Weakness:* when the mass is small or peripheral the heat can drift toward adjacent tissue.
* **Pituitary** — the hotspot is tightly localised on the central sellar / suprasellar region, the
  most anatomically consistent and therefore most reliable of the four classes.
* **No Tumor** — no focal *lesion* hotspot; attention spreads over normal brain structures rather
  than pointing at a single mass. This is the *correct* signature of "nothing abnormal to point at"
  rather than a failure to localise.

Across the three tumour classes the heat lands on a compact focal region rather than on the skull
or background, indicating the model keys on the lesion itself — and the high confidence on these
cases reinforces that the learned features are genuinely class-discriminative.

---

## Misclassification Analysis

The most *confident* mistakes are the most informative — they reveal systematic confusions
rather than random noise. Each row shows the original scan, the Grad-CAM heatmap, and the
overlay with predicted vs. actual class and confidence.

In [25]:
# Show the most confident errors (most informative). Falls back gracefully if there are few.
mis_idx = pick(correct=False, n=6)
n_err = int((pred_all != y_true).sum())
print(f"Total misclassified: {n_err} / {len(y_true)} "
      f"({100 * n_err / len(y_true):.1f}%). Showing the {len(mis_idx)} most confident errors.")
if len(mis_idx):
    gradcam_panel(mis_idx, "Grad-CAM — Misclassified Examples (most confident errors)")
else:
    print("No misclassified examples to display.")

# Which class pairs does the model actually confuse? (counted over every error)
err_mask = pred_all != y_true
pair_counts = Counter((CLASS_NAMES[t], CLASS_NAMES[p])
                      for t, p in zip(y_true[err_mask], pred_all[err_mask]))
print("\nMost confused class pairs (actual -> predicted):")
for (t, p), k in pair_counts.most_common(3):
    print(f"  {DISPLAY[t]:>11} -> {DISPLAY[p]:<11} : {k} cases")

Total misclassified: 240 / 1600 (15.0%). Showing the 6 most confident errors.


<Figure size 900x1800 with 18 Axes>


Most confused class pairs (actual -> predicted):
       Glioma -> Meningioma  : 81 cases
   Meningioma -> Pituitary   : 53 cases
       Glioma -> Pituitary   : 34 cases


**Likely reasons for these errors:**

* **Similar tumour appearance** — `glioma` and `meningioma` can share intensity, shape, and
  location cues, and the heatmaps often land on the same ambiguous region for both.
* **Ambiguous MRI patterns** — small, atypical, or partially-imaged lesions produce weaker,
  less localised activations.
* **Image quality / framing** — differences in contrast, slice orientation, or cropping can
  shift the model's attention away from the lesion.

Where the overlay highlights a plausible region but the label is still wrong, the issue is
*discrimination between similar classes*, not a failure to locate the tumour.

---

## Confidence Analysis

Softmax confidence is a useful (if imperfect) reliability signal. We compare the confidence of
correct vs. incorrect predictions and quantify how trustworthy high-confidence outputs are.

In [26]:
correct_mask = pred_all == y_true

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))
ax1.hist(conf_all[correct_mask],  bins=20, range=(0.25, 1.0), color="seagreen",
         alpha=0.7, label="correct", edgecolor="black")
ax1.hist(conf_all[~correct_mask], bins=20, range=(0.25, 1.0), color="indianred",
         alpha=0.7, label="incorrect", edgecolor="black")
ax1.set_title("Confidence distribution"); ax1.set_xlabel("max softmax probability")
ax1.set_ylabel("count"); ax1.legend()

ax2.boxplot([conf_all[correct_mask], conf_all[~correct_mask]])
ax2.set_xticks([1, 2]); ax2.set_xticklabels(["correct", "incorrect"])
ax2.set_title("Confidence: correct vs. incorrect"); ax2.set_ylabel("confidence")
plt.tight_layout(); plt.show()

# Reliability numbers.
hi, lo = conf_all >= 0.90, conf_all < 0.50
print(f"Mean confidence  — correct   : {conf_all[correct_mask].mean():.1%}")
print(f"Mean confidence  — incorrect : {conf_all[~correct_mask].mean():.1%}")
print(f"High-confidence (>=90%)      : {hi.sum():4d} preds "
      f"({100 * hi.mean():.1f}%), {100 * correct_mask[hi].mean() if hi.sum() else 0:.1f}% correct")
print(f"Low-confidence  (<50%)       : {lo.sum():4d} preds "
      f"({100 * lo.mean():.1f}%), {100 * correct_mask[lo].mean() if lo.sum() else 0:.1f}% correct")

<Figure size 1300x450 with 2 Axes>

Mean confidence  — correct   : 92.7%
Mean confidence  — incorrect : 70.6%
High-confidence (>=90%)      : 1076 preds (67.2%), 97.2% correct
Low-confidence  (<50%)       :   42 preds (2.6%), 40.5% correct


**Reading the reliability:** a well-behaved classifier is *more* confident when it is right than
when it is wrong — the green (correct) distribution should sit clearly to the right of the red
(incorrect) one, and high-confidence (≥90%) predictions should be overwhelmingly correct. This
supports a practical clinical workflow: **auto-accept high-confidence cases and route
low-confidence cases for human review**. Note that a raw softmax score is not *guaranteed* to be a
calibrated probability — and the model can still be confidently wrong (Phase 5 measures the actual
calibration error directly). That residual risk is exactly why the Grad-CAM visual check matters as
a second line of defence.

## Least-Confident Predictions

The reliability numbers above are aggregate; here we look at the individual cases the model is
*most unsure* about. These low-confidence scans are exactly the ones a triage workflow would
**route to a radiologist** rather than auto-accept. Their Grad-CAM maps are typically diffuse or
land on ambiguous tissue — a faithful visual signature of genuine uncertainty.

In [27]:
# The least-confident predictions across the whole test set (model is most uncertain here).
low_idx = np.argsort(conf_all)[:3]
print("Lowest test-set confidences: " +
      ", ".join(f"{conf_all[i]:.0%}" for i in low_idx))
gradcam_panel(low_idx, "Grad-CAM — Lowest-Confidence Predictions (route to human review)")

Lowest test-set confidences: 34%, 37%, 38%


<Figure size 900x900 with 9 Axes>

---

## Class-Level Explainability

A focused check across all four classes — **Glioma, Meningioma, Pituitary, No Tumor** — to ask:
does the model attend to a *meaningful* region for each class? One strong representative per
class is shown below.

In [28]:
# One strong (highest-confidence correct) Grad-CAM representative per class.
fig, axes = plt.subplots(2, 2, figsize=(8, 8))
for ax, c in zip(axes.ravel(), range(NUM_CLASSES)):
    sel = pick(true_c=c, correct=True, n=1)
    if len(sel):
        _, _, overlay, pred_i, probs = _gradcam_for_index(sel[0])
        ax.imshow(overlay)
        ax.set_title(f"{DISPLAY[CLASS_NAMES[c]]} — focus check ({probs[pred_i]:.0%})", fontsize=11)
    ax.axis("off")
plt.suptitle("Class-Level Explainability — Where the Model Looks", y=0.98, fontsize=13)
plt.tight_layout(); plt.show()

<Figure size 800x800 with 4 Axes>

**Focus check:** for `Glioma`, `Meningioma`, and `Pituitary`, the activation should sit on the
lesion region characteristic of that tumour type (for example, the central sellar region for
pituitary tumours). For `No Tumor`, the absence of a sharp focal hotspot is itself the correct
behaviour. If any class *consistently* highlighted the skull, image edges, or background, that
would flag a shortcut-learning / data-leakage risk worth investigating before clinical use.

---

## Explainable AI in Medical Imaging

Explainability is not a luxury in medical imaging — it is a prerequisite for safe deployment. A
model that predicts "glioma" with 95% confidence is clinically useless, and potentially
dangerous, if neither the radiologist nor the engineer can tell *why* it reached that
conclusion. Explainable AI (XAI) bridges the gap between predictive performance and clinical
accountability.

**Why transparency matters to clinicians.** Radiologists are ultimately responsible for
diagnoses; they cannot defer that responsibility to an opaque algorithm. To trust a model's
output, they need to verify that it is reasoning from medically relevant evidence — the lesion
itself — rather than from spurious correlations such as scanner artefacts, text annotations, or
patient-positioning cues. Visual explanations let a clinician quickly sanity-check each
prediction and decide whether to accept, question, or override it. This human-in-the-loop
oversight is what makes AI-assisted diagnosis defensible.

**Benefits of Grad-CAM.** Grad-CAM is attractive because it is model-agnostic across CNNs,
requires no architectural changes or retraining, and produces an intuitive heatmap that overlays
directly on the scan. It helps detect *shortcut learning* (e.g. a model keying on the skull
outline instead of the tumour), supports error analysis by revealing where attention fell on
misclassified cases, and provides a communication tool that non-engineers can interpret. In this
project it let us confirm that correct predictions concentrate on focal lesion regions, while
ambiguous glioma/meningioma cases share overlapping attention — a faithful explanation of the
confusion.

**Limitations.** Grad-CAM is coarse: heatmaps come from low-resolution feature maps and are
upsampled, so they indicate *approximate* regions, not precise boundaries. It is a post-hoc
explanation — it describes correlation with the output, not the true causal reasoning, and can
be unstable across similar inputs. It does not guarantee correctness: a model can attend to the
right region and still misclassify, or attend to the wrong region and be right by coincidence.
It explains *where*, never *why* a region is informative.

**Ethical considerations.** Explanations can create false reassurance — a plausible-looking
heatmap may lull clinicians into over-trusting a flawed model (automation bias). Models trained
on narrow datasets may not generalise across scanners, demographics, or populations, raising
fairness concerns. Patient privacy, informed consent for AI-assisted decisions, and clear
accountability when an AI contributes to an error all remain open responsibilities. XAI is a
necessary safeguard, but it must complement — never replace — rigorous clinical validation,
diverse training data, and human medical judgement.

---

# Phase 4 – AI Application & Deployment Preparation

Phases 1–3 built and *explained* the model; Phase 4 turns it into something a **user** can actually
interact with. We assemble a single-image prediction pipeline, simulate per-patient clinical
reports, auto-generate a structured diagnostic report, summarise performance in a dashboard, sketch
the end-to-end architecture, and discuss what real-world deployment would genuinely require.

Everything here **reuses the model and Grad-CAM helpers already defined above** — no retraining and
no new files, in keeping with the single-notebook design.

> ⚠️ **Disclaimer.** This is a research and educational demonstration only. It is **not** a medical
> device and must not be used for clinical decision-making.

## Single Image Prediction

A complete, user-facing inference workflow in one function. Given *any* MRI file it preprocesses the
image, runs the model, and returns a friendly result: the original scan, the Grad-CAM heatmap and
overlay, a probability bar for every class, and a plain-language diagnosis with confidence — exactly
what a non-technical user needs to see.

In [29]:
from IPython.display import display, Markdown

def predict_mri(path, show=True):
    """End-to-end single-image inference: preprocess -> predict -> Grad-CAM -> friendly result.
    Reuses load_image / make_gradcam_heatmap / colorize_and_overlay from Phase 3.
    Returns a dict with the prediction, confidence, probability vector, and rendered arrays."""
    arr = load_image(path)                                   # raw [0,255] float, model-ready
    heatmap, pred_i, probs = make_gradcam_heatmap(arr[None])
    img = arr.astype("uint8")
    colored, overlay = colorize_and_overlay(img, heatmap)
    pred_name, conf = DISPLAY[CLASS_NAMES[pred_i]], float(probs[pred_i])

    if show:
        fig = plt.figure(figsize=(13, 3.6))
        gs = fig.add_gridspec(1, 4)
        for k, (im, t) in enumerate([(img, "Original MRI"),
                                     (colored, "Grad-CAM heatmap"),
                                     (overlay, "Overlay")]):
            ax = fig.add_subplot(gs[0, k]); ax.imshow(im)
            ax.set_title(t, fontsize=10); ax.axis("off")
        axb = fig.add_subplot(gs[0, 3])
        order = np.argsort(probs)
        axb.barh([DISPLAY[CLASS_NAMES[i]] for i in order], probs[order],
                 color=["#d62728" if i == pred_i else "#c7c7c7" for i in order])
        axb.set_xlim(0, 1); axb.set_title("Class probabilities", fontsize=10)
        for i, v in zip(order, probs[order]):
            axb.text(min(v + 0.02, 0.85), DISPLAY[CLASS_NAMES[i]], f"{v:.0%}",
                     va="center", fontsize=9)
        for s in ["top", "right"]: axb.spines[s].set_visible(False)
        plt.suptitle(f"Diagnosis: {pred_name}  —  confidence {conf:.1%}", fontsize=13, y=1.05)
        plt.tight_layout(); plt.show()

        print(f"FINAL DIAGNOSIS : {pred_name}")
        print(f"CONFIDENCE      : {conf:.1%}")
        print(f"INTERPRETATION  : {auto_explanation(pred_i, pred_i, conf)}")

    return {"path": path, "pred": pred_name, "class": CLASS_NAMES[pred_i],
            "confidence": conf, "probs": probs, "image": img,
            "overlay": overlay, "pred_index": pred_i}

# Demo on a real MRI file (any path on disk works — this is the single-image entry point).
demo_dir  = os.path.join(TEST_DIR, "glioma")
demo_path = os.path.join(demo_dir, sorted(os.listdir(demo_dir))[0])
print("Single-image demo on:", os.path.relpath(demo_path, PROJECT_ROOT), "\n")
_ = predict_mri(demo_path)

Single-image demo on: Dataset\Testing\glioma\Te-gl_1.jpg 



<Figure size 1300x360 with 4 Axes>

FINAL DIAGNOSIS : No Tumor
CONFIDENCE      : 51.5%
INTERPRETATION  : The model predicted No Tumor with moderate confidence (52%); it attended to uniform tissue with no focal tumour region, though less decisively.


## Patient Case Analysis

A simulated clinical worklist: one representative scan per class, each presented as a short
**patient case** — MRI, Grad-CAM, predicted diagnosis, confidence, and an automatic interpretation.
This mirrors how a radiologist might review an AI second-opinion case by case.

In [30]:
# One representative (high-confidence, correct) scan per tumour type, as patient cases.
case_indices = [pick(true_c=c, correct=True, n=1)[0] for c in range(NUM_CLASSES)]

for n, idx in enumerate(case_indices, 1):
    img, colored, overlay, pred_i, probs = _gradcam_for_index(idx)
    true_i, conf = int(y_true[idx]), probs[pred_i]

    fig, ax = plt.subplots(1, 2, figsize=(6.2, 3.1))
    ax[0].imshow(img);     ax[0].set_title("MRI scan", fontsize=10);  ax[0].axis("off")
    ax[1].imshow(overlay); ax[1].set_title("Grad-CAM",  fontsize=10); ax[1].axis("off")
    plt.tight_layout(); plt.show()

    print(f"Patient Case #{n}")
    print(f"Prediction : {DISPLAY[CLASS_NAMES[pred_i]]}")
    print(f"Confidence : {conf:.1%}")
    print(f"Reference  : {DISPLAY[CLASS_NAMES[true_i]]} (dataset label)")
    print( "Model Interpretation:")
    print(f"  {auto_explanation(pred_i, true_i, conf)}")
    print("=" * 72)

<Figure size 620x310 with 2 Axes>

Patient Case #1
Prediction : Glioma
Confidence : 100.0%
Reference  : Glioma (dataset label)
Model Interpretation:
  The model confidently predicted Glioma (100%), focusing primarily on the highlighted brain region associated with characteristics of Glioma.


<Figure size 620x310 with 2 Axes>

Patient Case #2
Prediction : Meningioma
Confidence : 99.9%
Reference  : Meningioma (dataset label)
Model Interpretation:
  The model confidently predicted Meningioma (100%), focusing primarily on the highlighted brain region associated with characteristics of Meningioma.


<Figure size 620x310 with 2 Axes>

Patient Case #3
Prediction : No Tumor
Confidence : 100.0%
Reference  : No Tumor (dataset label)
Model Interpretation:
  The model confidently predicted No Tumor (100%), focusing primarily on uniform tissue with no focal tumour region.


<Figure size 620x310 with 2 Axes>

Patient Case #4
Prediction : Pituitary
Confidence : 100.0%
Reference  : Pituitary (dataset label)
Model Interpretation:
  The model confidently predicted Pituitary (100%), focusing primarily on the highlighted brain region associated with characteristics of Pituitary.


## Automated MRI Analysis Report

A structured, markdown-rendered report for a single scan — the kind of artefact an AI service would
emit per study. It bundles the predicted class, confidence band, model used, Grad-CAM findings, a
short interpretation, and a recommended action driven by the confidence level.

In [31]:
def diagnostic_report(source):
    """Generate a professional, markdown-formatted AI analysis report for one MRI.
    `source` may be a file path (str, external image) or a test-set index (int)."""
    if isinstance(source, str):
        r = predict_mri(source, show=False)
        pred_i, probs, conf = r["pred_index"], r["probs"], r["confidence"]
        img, overlay, ref = r["image"], r["overlay"], "— (external image)"
    else:
        img, colored, overlay, pred_i, probs = _gradcam_for_index(source)
        conf = float(probs[pred_i]); ref = DISPLAY[CLASS_NAMES[int(y_true[source])]]
    pred_name = DISPLAY[CLASS_NAMES[pred_i]]

    fig, ax = plt.subplots(1, 2, figsize=(5.6, 2.9))
    ax[0].imshow(img);     ax[0].set_title("MRI", fontsize=9);      ax[0].axis("off")
    ax[1].imshow(overlay); ax[1].set_title("Grad-CAM", fontsize=9); ax[1].axis("off")
    plt.tight_layout(); plt.show()

    band   = "HIGH" if conf >= 0.85 else "MODERATE" if conf >= 0.50 else "LOW"
    action = ("Suitable for automated triage with routine clinician confirmation." if conf >= 0.85
              else "Flag for clinician review." if conf >= 0.50
              else "Low confidence — mandatory specialist review.")
    ranked = ", ".join(f"{DISPLAY[CLASS_NAMES[i]]} {probs[i]:.0%}" for i in np.argsort(-probs))

    md = f'''### 🧾 Automated MRI Analysis Report

| Field | Value |
|---|---|
| **Predicted class** | {pred_name} |
| **Confidence** | {conf:.1%} ({band}) |
| **Model** | {BEST_MODEL_NAME} (transfer learning) |
| **Dataset reference label** | {ref} |
| **Explainability** | Grad-CAM on `{GC_CONV_NAME}` |

**Grad-CAM findings.** {auto_explanation(pred_i, pred_i, conf)}

**Recommended action.** {action}

*Class probabilities:* {ranked}

> ⚠️ Research demonstration only — not a medical device and not for clinical use.'''
    display(Markdown(md))

# Demo: a full report for the first patient case.
diagnostic_report(case_indices[0])

<Figure size 560x290 with 2 Axes>

### 🧾 Automated MRI Analysis Report

| Field | Value |
|---|---|
| **Predicted class** | Glioma |
| **Confidence** | 100.0% (HIGH) |
| **Model** | EfficientNetB0 (transfer learning) |
| **Dataset reference label** | Glioma |
| **Explainability** | Grad-CAM on `efficientnetb0` |

**Grad-CAM findings.** The model confidently predicted Glioma (100%), focusing primarily on the highlighted brain region associated with characteristics of Glioma.

**Recommended action.** Suitable for automated triage with routine clinician confirmation.

*Class probabilities:* Glioma 100%, Pituitary 0%, Meningioma 0%, No Tumor 0%

> ⚠️ Research demonstration only — not a medical device and not for clinical use.

## Model Performance Dashboard

A single at-a-glance view of how the two models compare on the held-out test set — a styled metrics
table plus a grouped bar chart of accuracy, precision, recall, and F1, with the best model called
out explicitly.

In [32]:
# Dashboard: styled comparison table + grouped metric bars (reads the Phase 2 `comparison` table).
metrics  = ["Accuracy", "Precision", "Recall", "F1 Score"]
best_row = comparison.sort_values("Accuracy", ascending=False).iloc[0]

display(comparison.style
        .format({m: "{:.3f}" for m in metrics})
        .set_caption("Model comparison — macro-averaged on the held-out test set")
        .highlight_max(subset=metrics, color="#c8e6c9", axis=0))

x, w = np.arange(len(metrics)), 0.38
fig, ax = plt.subplots(figsize=(9, 4.3))
for k, (_, row) in enumerate(comparison.iterrows()):
    bars = ax.bar(x + (k - 0.5) * w, [row[m] for m in metrics], w, label=row["Model"])
    ax.bar_label(bars, fmt="%.2f", fontsize=8, padding=2)
ax.set_xticks(x); ax.set_xticklabels(metrics)
ax.set_ylim(0, 1.08); ax.set_ylabel("score")
ax.set_title("Model Performance Dashboard", fontsize=13)
ax.legend(); ax.grid(axis="y", alpha=0.3)
for s in ["top", "right"]: ax.spines[s].set_visible(False)
plt.tight_layout(); plt.show()

print(f"BEST MODEL : {best_row['Model']}")
print(f"  Accuracy : {best_row['Accuracy']:.1%}   Precision: {best_row['Precision']:.3f}   "
      f"Recall: {best_row['Recall']:.3f}   F1: {best_row['F1 Score']:.3f}")

,Model,Accuracy,Precision,Recall,F1 Score
0,CNN,0.712,0.725,0.712,0.703
1,EfficientNetB0,0.850,0.855,0.850,0.844


<Figure size 900x430 with 1 Axes>

BEST MODEL : EfficientNetB0
  Accuracy : 85.0%   Precision: 0.855   Recall: 0.850   F1: 0.844


## Project Architecture

The complete pipeline at a glance — from raw scans to an explainable, report-generating prediction
system. The diagram is drawn with matplotlib (no extra dependencies).

In [33]:
import matplotlib.patches as mpatches

stages = ["MRI Dataset\n(4 classes, ~7,200 scans)",
          "Preprocessing\n(resize 224², tf.data + augmentation)",
          "Models\nCNN baseline + EfficientNetB0 (transfer)",
          "Evaluation\n(accuracy, precision/recall/F1, confusion)",
          "Explainability\n(Grad-CAM heatmaps + confidence)",
          "Prediction System\n(single-image inference)",
          "Automated AI Report\n(diagnosis + interpretation)"]

fig, ax = plt.subplots(figsize=(6.6, 9.2))
ax.set_xlim(0, 10); ax.set_ylim(0, len(stages) * 2); ax.axis("off")
colors = plt.cm.Blues(np.linspace(0.45, 0.85, len(stages)))
for k, (label, col) in enumerate(zip(reversed(stages), reversed(colors))):
    y = k * 2 + 0.4
    ax.add_patch(mpatches.FancyBboxPatch((2, y), 6, 1.2, boxstyle="round,pad=0.1",
                                         fc=col, ec="black", lw=1.2))
    ax.text(5, y + 0.6, label, ha="center", va="center", fontsize=10, weight="bold")
    if k < len(stages) - 1:
        ax.annotate("", xy=(5, y + 1.25), xytext=(5, y + 2.0),
                    arrowprops=dict(arrowstyle="-|>", lw=1.8, color="#333"))
ax.set_title("End-to-End Project Architecture", fontsize=13, weight="bold")
plt.tight_layout(); plt.show()

<Figure size 660x920 with 1 Axes>

## Deployment Considerations

A trained notebook is not a product. Turning this classifier into something a clinic could use
raises engineering, regulatory, and ethical questions that go well beyond test-set accuracy.

**Serving options.**
* **Streamlit / Gradio app** — the fastest route to a demo: a web page where a user uploads an MRI
  and gets back the predicted class, confidence, and Grad-CAM overlay produced by `predict_mri`.
  Ideal for portfolios and internal review; not a regulated device.
* **REST API** — wrap the model in **FastAPI** (or TensorFlow Serving) behind a `/predict` endpoint
  returning JSON. This decouples the model from any front-end and lets hospital systems call it
  programmatically. Containerise with **Docker** for reproducible deploys.
* **Cloud** — host the container on a managed service (AWS SageMaker, GCP Vertex AI, Azure ML) for
  autoscaling, GPU inference, versioned endpoints, and audit logging. EfficientNetB0 is light enough
  to run acceptably on CPU, keeping costs modest.

**Hospital integration.** Real clinical use means **DICOM** input (not JPG/PNG), integration with
**PACS/RIS** systems via standards such as **HL7/FHIR**, and fitting the radiologist's existing
workflow as a *second reader* rather than a replacement. Latency, uptime, and fail-safe behaviour
("uncertain — refer to specialist") become hard requirements.

**Data privacy & governance.** MRI scans are protected health information: deployment must comply
with **GDPR / HIPAA**, use de-identified data, encrypt data in transit and at rest, and keep access
logs. Cloud processing needs a clear data-handling and consent framework.

**Regulation.** A diagnostic tool is a medical device. Clinical deployment would require regulatory
clearance (e.g. **FDA / UKCA / CE-marking** under the EU MDR) backed by prospective clinical
validation — far beyond a single offline score.

**Monitoring.** Post-deployment, the model must be watched for **data drift** (new scanners,
protocols, populations) and **performance decay**: confidence-distribution dashboards, human-override
tracking, periodic re-validation, and a retraining pipeline triggered when drift is detected.

## Limitations

* **Dataset scope.** A single public dataset of 2-D slices from a limited set of scanners and
  populations. Real clinics see more vendors, protocols, artefacts, and rarer presentations, so
  test-set accuracy almost certainly *overstates* real-world performance.
* **2-D, not volumetric.** Radiologists read full 3-D volumes across sequences (T1, T2, FLAIR,
  contrast). This model sees one pre-selected slice, discarding spatial and multi-sequence context
  that is often diagnostic.
* **Class coverage.** Four classes only. A real tool must handle other tumour types, post-surgical
  change, and out-of-distribution inputs rather than forcing every scan into one of four buckets.
* **Confidence needs verifying, not assuming.** Raw softmax scores are not *guaranteed* to be
  calibrated probabilities; Phase 5 measures this directly and finds EfficientNet is in fact fairly
  well calibrated (ECE 0.044), while the CNN is not (0.129). Even so, a small tail of
  high-confidence errors remains (see the Confidence Analysis).
* **Explainability is approximate.** Grad-CAM shows *where*, not *why*; it is coarse, post-hoc, and
  can look plausible even when the prediction is wrong. It supports trust but does not guarantee
  correctness.
* **No clinical validation.** Performance is measured offline on held-out images, not in a
  prospective study with real patients and clinician outcomes.

## Future Improvements

* **Larger, multi-site data** — train and validate across multiple institutions and scanners to
  improve generalisation, with an explicit out-of-distribution / "refer" pathway.
* **Lift glioma recall** — the clearest measured weakness (0.65): targeted/ harder glioma examples,
  class-weighting, or focal loss to reduce missed gliomas.
* **Volumetric & multi-modal models** — use full 3-D volumes and multiple MRI sequences (plus
  clinical metadata) for context closer to how radiologists actually read scans.
* **Confidence calibration** — EfficientNet is already fairly well calibrated (ECE 0.044), so this
  is a marginal tweak for it but a meaningful one for the CNN (0.129); temperature scaling or deep
  ensembles would tighten it further and sharpen the auto-accept / refer threshold.
* **Model ensembling** — combine EfficientNet with other backbones (ConvNeXt, ViT) to lift accuracy
  and derive uncertainty from disagreement.
* **Higher-resolution explainability** — Grad-CAM++, Score-CAM, or attention maps for sharper, more
  faithful localisation.
* **Prospective clinical validation** — the essential step before any real use: a controlled study
  measuring impact on radiologist accuracy, speed, and patient outcomes.
* **Productionisation** — DICOM support, a FastAPI/Streamlit service, containerised cloud deployment,
  and a monitoring + retraining pipeline.

---
# Phase 5 – Advanced Model Analysis

**Why this phase exists.** Phases 1–4 build, explain, and "deploy" an accurate classifier. But in medical AI, *accuracy is necessary, not sufficient*. A model that scores 95% yet is **silently, confidently wrong** on the remaining 5% is dangerous in a clinic — the failures are exactly the cases no one double-checks. Phase 5 stress-tests the model the way a reviewer, clinician, or regulator would: it dissects **failures**, **confusion structure**, **per-class reliability**, **confidence calibration**, and **explanation quality**, then turns those diagnostics into research-grade conclusions.

**What we reuse (no retraining).** Everything below reuses the already-trained `cnn` and `effnet` models, the cached test predictions (`pred_all`, `conf_all`, `effnet_probs`), and the Phase 3 Grad-CAM toolkit (`gradcam_panel`, `pick`, `make_gradcam_heatmap`). The only new computation is the CNN's cached probabilities and a couple of lightweight, no-train experiments.

| # | Section | Question it answers |
|---|---------|---------------------|
| 1 | Error Analysis | *Where and how does the model fail?* |
| 2 | Class Confusion | *Which classes get mistaken for each other, and why?* |
| 3 | Per-Class Performance | *Is the model uniformly good, or weak on one tumour type?* |
| 4 | Confidence Analysis | *Does the model "know when it knows"?* |
| 5 | Calibration (ECE) | *Do the probabilities mean what they say?* |
| 6 | Explainability Evaluation | *Is Grad-CAM looking at the right evidence?* |
| 7 | CNN vs EfficientNet | *Which model, and why?* |
| 8 | Research Experiments | *What concretely improves reliability?* |
| 9 | Reliability & Trust | *What does responsible medical AI require?* |
| 10 | Phase 5 Summary | *What did we learn?* |

In [34]:
# ---- Phase 5 shared setup -------------------------------------------------
# EfficientNet predictions (effnet_probs / pred_all / conf_all) were cached in Phase 3.
# Here we add the CNN's softmax outputs so the two models can be compared head-to-head,
# and define convenience masks used throughout Phase 5.
cnn_probs   = cnn.predict(test_ds, verbose=0)
cnn_conf    = cnn_probs.max(axis=1)
cnn_pred_p5 = cnn_probs.argmax(axis=1)
assert np.array_equal(cnn_pred_p5, cnn_pred), "CNN prediction mismatch vs. Phase 2"

correct_mask = pred_all == y_true          # best (EfficientNet) model, per-image correctness
err_mask     = ~correct_mask
err_idx      = np.where(err_mask)[0]

print(f"Phase 5 ready — {BEST_MODEL_NAME}: {err_mask.sum()} errors / {len(y_true)} test "
      f"images ({100*err_mask.mean():.1f}%). CNN probabilities cached for comparison.")

Phase 5 ready — EfficientNetB0: 240 errors / 1600 test images (15.0%). CNN probabilities cached for comparison.


## Phase 5 – Error Analysis

Failures are the most information-dense part of any evaluation. We rank every misclassification by the confidence the model placed in its **wrong** answer, then inspect the two extremes:

* **Most-confident errors** — the *high-risk* failures: the model is sure and wrong, so a human would be least likely to question it.
* **Least-confident errors** — the model was already hesitant; a confidence-thresholded system would correctly flag these for human review.

Each example shows the original MRI, the actual vs. predicted label, the confidence, and the Grad-CAM heatmap so we can see *what the model looked at* when it slipped.

In [35]:
# Rank every error by the model's confidence in its (incorrect) prediction.
err_sorted = err_idx[np.argsort(-conf_all[err_idx])]   # most-confident error first
if len(err_sorted):
    print(f"{BEST_MODEL_NAME}: {len(err_sorted)} errors.  "
          f"Most-confident error: {conf_all[err_sorted[0]]:.0%}  |  "
          f"Least-confident error: {conf_all[err_sorted[-1]]:.0%}")

    # (a) Most-confident mistakes — the dangerous ones.
    gradcam_panel(err_sorted[:3],
                  "Error Analysis — Most CONFIDENT Incorrect Predictions (high-risk failures)")
else:
    print("No misclassified examples on the test set.")

EfficientNetB0: 240 errors.  Most-confident error: 99%  |  Least-confident error: 34%


<Figure size 900x900 with 9 Axes>

In [36]:
# (b) Least-confident mistakes — the model already signalled uncertainty here, so a
#     confidence threshold would route these scans to a radiologist (a 'safe' failure mode).
if len(err_sorted):
    gradcam_panel(err_sorted[::-1][:3],
                  "Error Analysis — Least CONFIDENT Incorrect Predictions (uncertain failures)")

<Figure size 900x900 with 9 Axes>

In [37]:
# Clinically the errors are NOT equal: a tumour read as 'No Tumor' (false negative) is the
# highest-risk mistake, while a healthy scan read as a tumour (false positive) merely triggers
# a confirmatory scan. We quantify both directions explicitly.
notumor_id  = CLASS_NAMES.index("notumor")
tumor_true  = y_true != notumor_id
missed      = tumor_true & (pred_all == notumor_id)               # tumour -> No Tumor (FN)
false_alarm = (y_true == notumor_id) & (pred_all != notumor_id)   # No Tumor -> tumour (FP)

print(f"MISSED TUMOURS (tumour predicted 'No Tumor') : {missed.sum():3d} / {tumor_true.sum()} "
      f"tumour scans  ({100*missed.sum()/tumor_true.sum():.1f}% false-negative rate)")
for c in range(NUM_CLASSES):
    if c == notumor_id:
        continue
    m = (y_true == c) & (pred_all == notumor_id)
    print(f"      from {DISPLAY[CLASS_NAMES[c]]:<11}: {int(m.sum())}")
print(f"FALSE ALARMS  ('No Tumor' predicted tumour)  : {false_alarm.sum():3d} / "
      f"{int((y_true==notumor_id).sum())} healthy scans  "
      f"({100*false_alarm.sum()/(y_true==notumor_id).sum():.1f}% false-positive rate)")

tumor_classes = [c for c in range(NUM_CLASSES) if c != notumor_id]
counts = [int(((y_true == c) & (pred_all == notumor_id)).sum()) for c in tumor_classes]
fig, ax = plt.subplots(figsize=(7, 3.2))
bars = ax.barh([DISPLAY[CLASS_NAMES[c]] for c in tumor_classes], counts, color="firebrick")
ax.bar_label(bars, padding=3)
ax.set_xlabel("scans missed as 'No Tumor' (false negatives)")
ax.set_title("Missed-Tumour Analysis - the highest-risk error mode")
ax.margins(x=0.12)
for s in ["top", "right"]:
    ax.spines[s].set_visible(False)
plt.tight_layout(); plt.show()

MISSED TUMOURS (tumour predicted 'No Tumor') :  42 / 1200 tumour scans  (3.5% false-negative rate)
      from Glioma     : 27
      from Meningioma : 15
      from Pituitary  : 0
FALSE ALARMS  ('No Tumor' predicted tumour)  :   4 / 400 healthy scans  (1.0% false-positive rate)


<Figure size 700x320 with 1 Axes>

**Observations — grounded in the actual errors.**

* **Glioma is the bottleneck class.** EfficientNet's glioma recall is just **0.65** (vs. 0.78 / 0.99 / 0.99 for meningioma / no-tumor / pituitary) — **~35% of gliomas are misclassified**, and almost every confident error in the panels above is a *true glioma*. High glioma *precision* (0.91) with low *recall* means the model is conservative about calling "glioma" and leaks those cases into other classes.
* **The dangerous failures are the missed tumours.** The single most clinically serious error is a real tumour read as **No Tumor** (a false negative). The next cell quantifies this directly — glioma alone contributes 27 such cases. The most-confident error on the whole test set sits at **99%** confidence, exactly the kind of mistake a deployed system would act on without hesitation.
* **Confident errors fall on visually overlapping tumour types.** Glioma->meningioma (81 cases) dominates: on a single 2-D slice these share intensity, mass effect, and location, and the Grad-CAM overlays confirm attention *is* on a lesion — the model found real evidence but attributed it to the wrong class.
* **Uncertain errors are genuine edge cases** (atypical or off-centre slices, unusual contrast). Their low confidence is a feature, not a bug: the confidence threshold in Section 8 catches and defers them, so they are the "safe" failure mode.
* **Takeaway:** shrinking the *confident-and-wrong* tail — especially missed tumours — matters far more than shaving tenths off headline accuracy.

## Class Confusion Analysis

The confusion matrix tells us *which* classes the model trades off. We row-normalise it (each row sums to 1, so the diagonal is per-class **recall**) and rank the off-diagonal cells to expose the dominant confusion pairs and their rates.

In [38]:
from sklearn.metrics import confusion_matrix

cm      = confusion_matrix(y_true, pred_all, labels=range(NUM_CLASSES))
cm_norm = cm / cm.sum(axis=1, keepdims=True)        # row-normalised -> per-true-class rates

# Rank off-diagonal (true != pred) confusions.
pairs = [(t, p, cm[t, p], cm_norm[t, p])
         for t in range(NUM_CLASSES) for p in range(NUM_CLASSES)
         if t != p and cm[t, p] > 0]
pairs.sort(key=lambda r: -r[2])

print("Most confused class pairs (true -> predicted):")
for t, p, n, rate in pairs[:5]:
    print(f"  {DISPLAY[CLASS_NAMES[t]]:>11} -> {DISPLAY[CLASS_NAMES[p]]:<11} : "
          f"{n:2d} cases  ({rate:.1%} of all {DISPLAY[CLASS_NAMES[t]]} scans)")

Most confused class pairs (true -> predicted):
       Glioma -> Meningioma  : 81 cases  (20.2% of all Glioma scans)
   Meningioma -> Pituitary   : 53 cases  (13.2% of all Meningioma scans)
       Glioma -> Pituitary   : 34 cases  (8.5% of all Glioma scans)
       Glioma -> No Tumor    : 27 cases  (6.8% of all Glioma scans)
   Meningioma -> Glioma      : 22 cases  (5.5% of all Meningioma scans)


In [39]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Left: row-normalised confusion matrix (diagonal = recall).
im = ax1.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
ax1.set_xticks(range(NUM_CLASSES)); ax1.set_yticks(range(NUM_CLASSES))
ax1.set_xticklabels([DISPLAY[c] for c in CLASS_NAMES], rotation=45, ha="right")
ax1.set_yticklabels([DISPLAY[c] for c in CLASS_NAMES])
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        ax1.text(j, i, f"{cm_norm[i, j]:.0%}", ha="center",
                 color="white" if cm_norm[i, j] > 0.5 else "black", fontsize=9)
ax1.set_title(f"{BEST_MODEL_NAME} — Row-Normalised Confusion\n(diagonal = recall)")
ax1.set_xlabel("Predicted"); ax1.set_ylabel("True")
fig.colorbar(im, ax=ax1, fraction=0.046, pad=0.04)

# Right: top confused pairs as confusion-rate bars.
top    = pairs[:6]
labels = [f"{DISPLAY[CLASS_NAMES[t]]}->{DISPLAY[CLASS_NAMES[p]]}" for t, p, _, _ in top]
rates  = [r for *_, r in top]
ax2.barh(range(len(top)), rates, color="indianred")
ax2.set_yticks(range(len(top))); ax2.set_yticklabels(labels)
ax2.invert_yaxis(); ax2.set_xlabel("confusion rate (share of true class)")
ax2.set_title("Top Confused Class Pairs")
for i, r in enumerate(rates):
    ax2.text(r + 0.002, i, f"{r:.1%}", va="center", fontsize=9)
plt.tight_layout(); plt.show()

<Figure size 1300x500 with 3 Axes>

**Discussion — what the matrix actually shows.**

* **The confusion is asymmetric, and glioma is the source.** The dominant pairs are **glioma->meningioma (81 cases, 20% of all gliomas)**, **meningioma->pituitary (53, 13%)**, and **glioma->pituitary (34, 9%)**. Note the asymmetry: glioma->meningioma (81) vastly outnumbers meningioma->glioma (22). Glioma is the hardest class to pin down, not a symmetric glioma/meningioma mix-up.
* **Similar visual characteristics.** On a single 2-D slice, glioma, meningioma, and pituitary lesions can present with comparable signal intensity, oedema, and mass effect; the most discriminative 3-D cues (dural tail, infiltration pattern, sellar location) are simply not in the image the model sees.
* **`No Tumor` is the cleanest class** — recall **0.99** and very few confusions — a reassuring sign the model learned "lesion vs. no lesion", not a texture shortcut. But note the *direction* that does occur: a handful of true tumours still leak into No Tumor (quantified in the Error-Analysis section), and those are the costly ones.
* **Why it matters clinically.** A glioma->meningioma error (wrong tumour type, still flagged for follow-up) and a tumour->No-Tumor error (missed entirely) carry very different risk. Equal-looking off-diagonal counts hide unequal clinical cost, which is why we pair the matrix with the missed-tumour audit and calibrated abstention (Sections 5, 8).

## Class-Level Evaluation

Headline accuracy averages over classes and can mask a weak one. Here we compute **precision, recall, F1, and support** for every tumour type and visualise them side by side.

In [40]:
from sklearn.metrics import classification_report

rep = classification_report(y_true, pred_all, labels=range(NUM_CLASSES),
                            target_names=[DISPLAY[c] for c in CLASS_NAMES],
                            output_dict=True)
per_class = pd.DataFrame({DISPLAY[c]: rep[DISPLAY[c]] for c in CLASS_NAMES}).T
per_class = per_class[["precision", "recall", "f1-score", "support"]]

display(per_class.style
        .format({"precision": "{:.3f}", "recall": "{:.3f}",
                 "f1-score": "{:.3f}", "support": "{:.0f}"})
        .background_gradient(subset=["precision", "recall", "f1-score"],
                             cmap="Greens", vmin=0.70, vmax=1.0)
        .set_caption(f"{BEST_MODEL_NAME} — Per-Class Performance (held-out test set)"))

,precision,recall,f1-score,support
Glioma,0.915,0.645,0.757,400
Meningioma,0.781,0.775,0.778,400
No Tumor,0.904,0.990,0.945,400
Pituitary,0.820,0.990,0.897,400


In [41]:
metrics3 = ["precision", "recall", "f1-score"]
x, w = np.arange(NUM_CLASSES), 0.25
fig, ax = plt.subplots(figsize=(9, 4.5))
for k, m in enumerate(metrics3):
    bars = ax.bar(x + (k - 1) * w, per_class[m].values, w, label=m)
    ax.bar_label(bars, fmt="%.2f", fontsize=7, padding=2)
ax.set_xticks(x); ax.set_xticklabels([DISPLAY[c] for c in CLASS_NAMES])
ax.set_ylim(0, 1.10); ax.set_ylabel("score")
ax.set_title(f"{BEST_MODEL_NAME} — Per-Class Precision / Recall / F1")
ax.legend(loc="lower right"); ax.grid(axis="y", alpha=0.3)
for s in ["top", "right"]:
    ax.spines[s].set_visible(False)
plt.tight_layout(); plt.show()

<Figure size 900x450 with 1 Axes>

**Interpretation — read against the numbers.**

* **Glioma is the weak class: recall 0.65 with precision 0.91.** That precision/recall split is diagnostic — the model rarely calls "glioma" wrongly, but when a glioma *is* present it misses about a third of them, leaking them into meningioma / pituitary / no-tumor. Because **recall is the safety-critical metric for a tumour class** (a missed tumour is far worse than a false alarm a radiologist later clears), glioma recall is the single most important number to improve.
* **Pituitary and No Tumor are near-ceiling** (recall 0.99 each), and **meningioma is balanced** (0.78 / 0.78). So the headline 85% accuracy is not propped up by one easy class — it is dragged *down* by glioma specifically.
* **The per-class F1 dip and the Section-2 confusion hot-spot agree:** both point at glioma, a consistency check that the weakness is real and not a metric artefact.
* **Balanced support (400 per class)** means macro-averaging is fair here — no class is rare enough for its score to be statistical noise.

## Prediction Confidence Analysis

A trustworthy classifier should be **more confident when it is right than when it is wrong**. We look at the full confidence distribution, compare correct vs. incorrect predictions, and — most importantly — measure whether accuracy actually rises with confidence.

In [42]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))

# (1) Overall confidence histogram.
axes[0].hist(conf_all, bins=20, range=(0.25, 1.0), color="steelblue",
             edgecolor="black", alpha=0.85)
axes[0].axvline(conf_all.mean(), color="navy", ls="--", label=f"mean {conf_all.mean():.0%}")
axes[0].set_title("All predictions"); axes[0].set_xlabel("max softmax probability")
axes[0].set_ylabel("count"); axes[0].legend()

# (2) Correct vs incorrect, density-normalised so the shapes are comparable.
axes[1].hist(conf_all[correct_mask], bins=20, range=(0.25, 1.0), density=True,
             color="seagreen", alpha=0.6, label="correct", edgecolor="black")
axes[1].hist(conf_all[err_mask], bins=20, range=(0.25, 1.0), density=True,
             color="indianred", alpha=0.6, label="incorrect", edgecolor="black")
axes[1].set_title("Correct vs incorrect (normalised)")
axes[1].set_xlabel("confidence"); axes[1].set_ylabel("density"); axes[1].legend()

# (3) Accuracy within confidence bands — does higher confidence really mean 'more right'?
bands = [(0.25, 0.5), (0.5, 0.7), (0.7, 0.9), (0.9, 1.0)]
acc_band, n_band, labels = [], [], []
for lo, hi in bands:
    m = (conf_all >= lo) & (conf_all < hi) if hi < 1.0 else (conf_all >= lo)
    acc_band.append(correct_mask[m].mean() if m.sum() else 0.0)
    n_band.append(int(m.sum())); labels.append(f"{lo:.0%}-{hi:.0%}")
bars = axes[2].bar(range(len(bands)), acc_band, color="mediumpurple", edgecolor="black")
axes[2].set_xticks(range(len(bands))); axes[2].set_xticklabels(labels)
axes[2].set_ylim(0, 1.10); axes[2].set_title("Accuracy by confidence band")
axes[2].set_ylabel("accuracy")
for b, n in zip(bars, n_band):
    axes[2].text(b.get_x() + b.get_width() / 2, b.get_height() + 0.02,
                 f"n={n}", ha="center", fontsize=8)
plt.tight_layout(); plt.show()

# High-confidence error audit — the cases that matter most in medical AI.
hi_conf = conf_all >= 0.90
hce     = hi_conf & err_mask
print(f"High-confidence (>=90%) predictions : {hi_conf.sum():4d}  "
      f"({100*correct_mask[hi_conf].mean():.1f}% correct)")
print(f"High-confidence ERRORS  (>=90% wrong): {hce.sum():4d}  "
      f"({100*hce.mean():.2f}% of all test images)")

<Figure size 1500x420 with 3 Axes>

High-confidence (>=90%) predictions : 1076  (97.2% correct)
High-confidence ERRORS  (>=90% wrong):   30  (1.88% of all test images)


**Discussion — is the confidence trustworthy?**

* **The two distributions separate.** The correct-prediction mass sits clearly to the right (high confidence) while the incorrect mass leans left — the model is, on average, more confident when right. This is the *minimum* property a usable confidence signal must have.
* **Accuracy rises monotonically with the confidence band.** The right-hand panel is the practical payoff: predictions above 90% confidence are correct far more often than those below 50%. That monotonicity is what lets us build a safe abstention rule (Section 8).
* **But the high-confidence error tail is the real risk.** Even a small percentage of confident mistakes is what a deployed system would act on blindly. Counting them explicitly (printed above) is more honest than reporting mean confidence alone.
* **Confidence ≠ correctness.** A high softmax probability is the model's *self-assessed* certainty, not a calibrated probability of being right — which is exactly what Section 5 measures.

## Reliability & Calibration Analysis

**What calibration means.** When a well-calibrated model says "90% confident", it should be right about 90% of the time. **Confidence and accuracy should match.** Deep networks are notoriously *over-confident* — they output 99% probabilities far more often than they are 99% right.

**Why it matters in medical AI.** A clinician using the model needs the confidence number to be *meaningful*: a calibrated 80% lets them reason about risk and decide when to order a second opinion. An over-confident model quietly removes that safety margin.

We draw a **reliability diagram** (accuracy vs. confidence per bin; the diagonal is perfect calibration) and compute the **Expected Calibration Error (ECE)** — the average gap between confidence and accuracy, weighted by how many predictions fall in each bin.

In [43]:
def compute_ece(conf, correct, n_bins=10):
    """Expected Calibration Error + per-bin stats. correct is a 0/1 array."""
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    ece, xs, accs, confs, counts = 0.0, [], [], [], []
    for i in range(n_bins):
        lo, hi = edges[i], edges[i + 1]
        m = (conf > lo) & (conf <= hi)
        if m.sum():
            acc_b, conf_b = correct[m].mean(), conf[m].mean()
            ece += (m.sum() / len(conf)) * abs(acc_b - conf_b)
            xs.append((lo + hi) / 2); accs.append(acc_b)
            confs.append(conf_b); counts.append(int(m.sum()))
    return ece, np.array(xs), np.array(accs), np.array(confs), np.array(counts)

ece_eff, xs, accs, confs, counts = compute_ece(conf_all, correct_mask.astype(float))
ece_cnn, *_ = compute_ece(cnn_conf, (cnn_pred_p5 == y_true).astype(float))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Reliability diagram.
ax1.plot([0, 1], [0, 1], "k--", label="perfect calibration")
ax1.bar(confs, accs, width=0.09, alpha=0.25, color="indianred")
ax1.plot(confs, accs, "o-", color="indianred", label=BEST_MODEL_NAME)
ax1.set_xlim(0, 1); ax1.set_ylim(0, 1)
ax1.set_xlabel("mean predicted confidence"); ax1.set_ylabel("actual accuracy")
ax1.set_title(f"Reliability Diagram (ECE = {ece_eff:.3f})")
ax1.legend(loc="upper left")

# Per-bin gap = confidence - accuracy (over/under-confidence).
gap    = confs - accs
colors = ["indianred" if g > 0 else "seagreen" for g in gap]
ax2.bar(range(len(xs)), gap, color=colors)
ax2.axhline(0, color="black", lw=0.8)
ax2.set_xticks(range(len(xs))); ax2.set_xticklabels([f"{x:.0%}" for x in xs], rotation=45)
ax2.set_title("Confidence - Accuracy per bin\n(red = over-confident, green = under-confident)")
ax2.set_xlabel("confidence bin"); ax2.set_ylabel("gap")
plt.tight_layout(); plt.show()

print(f"Expected Calibration Error — {BEST_MODEL_NAME:<14}: {ece_eff:.3f}")
print(f"Expected Calibration Error — CNN baseline   : {ece_cnn:.3f}")
print("(Lower is better; 0 means predicted confidence exactly matches observed accuracy.)")

<Figure size 1300x500 with 2 Axes>

Expected Calibration Error — EfficientNetB0: 0.044
Expected Calibration Error — CNN baseline   : 0.129
(Lower is better; 0 means predicted confidence exactly matches observed accuracy.)


**Calibration discussion — what the numbers say.**

* **EfficientNet is already fairly well calibrated: ECE = 0.044.** Its mean confidence (~89%) sits only ~4 points above its accuracy (85%) — *mildly* over-confident, not the gross over-confidence often assumed for deep nets. The **CNN baseline is the poorly calibrated one (ECE = 0.129)**, almost 3x worse, so here calibration tracks model quality, not just depth.
* **ECE is a single, comparable honesty score.** It ranks models independently of accuracy: a model can be more accurate yet worse calibrated. Here EfficientNet wins on *both* axes, which strengthens the deployment case in Section 7.
* **Calibration is still worth tightening.** Post-hoc **temperature scaling** (dividing logits by a learned scalar) typically trims residual ECE without changing a single prediction — marginal for EfficientNet, but it would materially help the CNN. We already see a free improvement from test-time augmentation in Section 8 (ECE 0.044 -> 0.036).
* **In a clinical pipeline** the *calibrated* confidence — not the argmax — drives the abstention threshold in Section 8. A well-calibrated 90% is what makes "auto-decide above 90%, else defer" a defensible policy.

## Explainability Assessment

Grad-CAM is only useful if the highlighted region is *clinically meaningful*. We sort the explanations into three buckets and inspect them, then quantify how *focused* the attention maps are.

* **Good** — high-confidence, correct, attention tight on a lesion.
* **Ambiguous** — correct but low-confidence; attention often diffuse.
* **Weak** — confident but wrong; attention landed on misleading evidence.

In [44]:
good_idx = pick(correct=True, n=2)                          # top-confidence correct overall
c_ok     = np.where(correct_mask)[0]
amb_idx  = c_ok[np.argsort(conf_all[c_ok])][:2]            # lowest-confidence *correct*
weak_idx = err_sorted[:2] if len(err_sorted) else []      # most-confident errors

gradcam_panel(good_idx, "Explainability — GOOD (confident, correct, focal attention)")
gradcam_panel(amb_idx,  "Explainability — AMBIGUOUS (correct but low-confidence / diffuse)")
if len(weak_idx):
    gradcam_panel(weak_idx, "Explainability — WEAK (confident but wrong: misplaced attention)")

<Figure size 900x600 with 6 Axes>

<Figure size 900x600 with 6 Axes>

<Figure size 900x600 with 6 Axes>

In [45]:
# Does the model's attention land more *tightly* when it is right? We measure 'attention
# concentration' = share of Grad-CAM activation mass held by the top 20% most-active pixels
# (heatmaps are max-normalised, so this captures spatial focus, not confidence magnitude).
#
# HYPOTHESIS TEST: a naive correct-vs-incorrect comparison might be confounded if 'No Tumor'
# scans (lesion-free, mostly correct) had diffuse maps. We therefore also report the comparison
# restricted to TUMOUR scans, plus No Tumor on its own, to check whether that confound is real.
# (The markdown below interprets the result — which is not what you might expect.)
def attention_concentration(idx):
    hm, _, _ = make_gradcam_heatmap(all_test_images[idx][None].astype("float32"))
    flat = np.sort(hm.ravel())[::-1]
    k = max(1, int(0.20 * len(flat)))
    return flat[:k].sum() / (flat.sum() + 1e-8)

rng = np.random.default_rng(SEED)
def mean_conc(idxs, cap=40):
    idxs = np.asarray(idxs)
    if len(idxs) == 0:
        return float("nan")
    sel = idxs if len(idxs) <= cap else rng.choice(idxs, size=cap, replace=False)
    return float(np.mean([attention_concentration(i) for i in sel]))

notumor_id = CLASS_NAMES.index("notumor")
tumor_mask = y_true != notumor_id
conc_all_corr   = mean_conc(np.where(correct_mask)[0])
conc_all_err    = mean_conc(np.where(err_mask)[0])
conc_corr_tumor = mean_conc(np.where(correct_mask & tumor_mask)[0])
conc_err_tumor  = mean_conc(np.where(err_mask & tumor_mask)[0])
conc_corr_notum = mean_conc(np.where(correct_mask & (y_true == notumor_id))[0])

print("Grad-CAM attention concentration (mass in top-20% of pixels):")
print(f"  naive  - all correct      : {conc_all_corr:.1%}")
print(f"  naive  - all incorrect    : {conc_all_err:.1%}   <- counter-intuitively HIGHER")
print( "  --- testing the No-Tumor confound (restrict to tumour scans) ---")
print(f"  tumour scans, correct     : {conc_corr_tumor:.1%}")
print(f"  tumour scans, incorrect   : {conc_err_tumor:.1%}   <- still higher: confound rejected")
print(f"  No-Tumor scans, correct   : {conc_corr_notum:.1%}   <- not diffuse (lesion-free ref)")

Grad-CAM attention concentration (mass in top-20% of pixels):
  naive  - all correct      : 48.7%
  naive  - all incorrect    : 58.6%   <- counter-intuitively HIGHER
  --- testing the No-Tumor confound (restrict to tumour scans) ---
  tumour scans, correct     : 48.4%
  tumour scans, incorrect   : 61.5%   <- still higher: confound rejected
  No-Tumor scans, correct   : 52.0%   <- not diffuse (lesion-free ref)


**Evaluation & limitations of Grad-CAM.**

* **Spatial focus does *not* track correctness — a hypothesis tested and rejected.** Measured over
  *all* predictions, incorrect cases were **more** concentrated than correct ones (≈59% vs ≈49% of
  mass in the top-20% pixels) — the opposite of the intuitive expectation. We hypothesised a
  `No Tumor` confound (lesion-free scans → diffuse maps → deflating the mostly-correct group) and
  tested it by restricting to tumour scans. **The data rejects that explanation:** within tumour
  scans the inversion *persists* (correct ≈48% vs incorrect ≈62%), and No-Tumor maps are not diffuse
  at all (≈52%, the most concentrated correct group). The honest conclusion is that **spatial
  concentration is not a reliability signal** — if anything it inversely correlates, because a
  *confident error* tends to fixate tightly on a specific but misleading region. The lesson for
  explanation-quality metrics: test the obvious confound, and never assume focus implies correctness.
* **Grad-CAM is coarse.** Computed at the last conv feature map, its native resolution is low
  (e.g. 7×7 for EfficientNet) and upsampled — it localises a *region*, not a boundary, and cannot
  delineate a tumour on its own.
* **It explains the predicted class, not correctness.** The "weak" examples above show clean
  heatmaps over misleading tissue: explainability reveals *where* the model looked, never *whether*
  it was right. Heatmap and calibrated confidence must be read together.
* **It is sensitive to target layer and gradient saturation.** Different layers yield different
  maps. Complementary methods (Grad-CAM++, Score-CAM, occlusion / SHAP) would triangulate the
  explanation and are a natural next step.
* **Bottom line:** Grad-CAM is a strong sanity-check and communication tool, but a *hypothesis*
  about model reasoning — not a clinical segmentation and not a guarantee of correctness.

## Model Comparison Study

We put the from-scratch **CNN baseline** and the transfer-learning **EfficientNetB0** side by side — not only on accuracy and F1, but on the *behavioural* axes that decide real-world trust: confidence separation, calibration (ECE), and the high-confidence error rate.

In [46]:
def model_diag(name, probs):
    pred = probs.argmax(1); conf = probs.max(1); corr = (pred == y_true)
    ece, *_ = compute_ece(conf, corr.astype(float))
    return {
        "Model": name,
        "Accuracy":       accuracy_score(y_true, pred),
        "Macro F1":       f1_score(y_true, pred, average="macro"),
        "Mean conf":      conf.mean(),
        "Conf | correct": conf[corr].mean(),
        "Conf | wrong":   conf[~corr].mean() if (~corr).any() else float("nan"),
        "ECE":            ece,
        "Hi-conf err %":  100 * ((conf >= 0.90) & ~corr).mean(),
    }

compare5 = pd.DataFrame([model_diag("CNN baseline",   cnn_probs),
                         model_diag("EfficientNetB0", effnet_probs)]).set_index("Model")
display(compare5.style
        .format({"Accuracy": "{:.3f}", "Macro F1": "{:.3f}", "Mean conf": "{:.1%}",
                 "Conf | correct": "{:.1%}", "Conf | wrong": "{:.1%}",
                 "ECE": "{:.3f}", "Hi-conf err %": "{:.2f}"})
        .set_caption("CNN vs EfficientNetB0 — accuracy, confidence behaviour, and calibration"))

,Accuracy,Macro F1,Mean conf,Conf | correct,Conf | wrong,ECE,Hi-conf err %
Model,,,,,,,
CNN baseline,0.712,0.703,84.0%,88.4%,73.2%,0.129,6.00
EfficientNetB0,0.850,0.844,89.4%,92.7%,70.6%,0.044,1.88


**Discussion — strengths, weaknesses, and the verdict.**

**EfficientNetB0 (transfer learning)**
* *Strengths:* it wins on **every** axis that matters — accuracy **0.85 vs 0.71**, macro-F1 **0.84 vs 0.70**, calibration **ECE 0.044 vs 0.129**, and a wider confidence gap between right (~93%) and wrong (~71%). ImageNet pre-training supplies rich low-level filters a few-thousand-image medical set cannot learn from scratch.
* *Weaknesses:* a larger, heavier, less transparent backbone; still *mildly* over-confident (mean confidence ~89% vs 85% accuracy), though far less so than the CNN.

**CNN baseline (from scratch)**
* *Strengths:* small, fast, fully transparent, and a genuine control — it proves the data is learnable without external weights and isolates the *contribution of transfer learning* (a +14-point accuracy jump).
* *Weaknesses:* materially lower accuracy / F1 and **3x worse calibration**; its glioma and meningioma recall (0.71 / 0.45) are weak enough to be clinically unusable.

**Which would I deploy?** **EfficientNetB0**, decisively — it dominates on accuracy *and* calibration, so there is no accuracy-vs-honesty trade-off to weigh. The one caveat is its glioma recall (0.65), which the abstention policy (Section 8) is designed to backstop. The CNN remains a valuable lightweight baseline and sanity check, not a production candidate.

## Experimental Findings

Rather than expensive retraining, we run two **lightweight, reproducible experiments** that directly probe reliability, and we summarise the architectural findings already established earlier in the notebook.

In [47]:
# Experiment A — Selective prediction (risk-coverage trade-off).
# If the model may *abstain* below a confidence threshold (routing those scans to a
# radiologist), how does accuracy on the cases it DOES decide change with coverage?
ths = np.linspace(0.50, 0.995, 40)
cov, acc_ret = [], []
for t in ths:
    keep = conf_all >= t
    if keep.sum():
        cov.append(keep.mean()); acc_ret.append(correct_mask[keep].mean())

fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.plot(cov, acc_ret, "o-", color="darkorange", ms=4)
ax.set_xlabel("coverage (fraction of scans auto-decided)")
ax.set_ylabel("accuracy on auto-decided scans")
ax.set_title("Experiment A — Selective Prediction (accuracy vs coverage)")
ax.invert_xaxis(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

for t in (0.90, 0.95):
    keep = conf_all >= t
    print(f"  threshold {t:.0%}: auto-decide {keep.mean():.0%} of scans at "
          f"{correct_mask[keep].mean():.1%} accuracy; "
          f"{int((~keep).sum())} scans routed to human review.")

<Figure size 750x450 with 1 Axes>

  threshold 90%: auto-decide 67% of scans at 97.2% accuracy; 524 scans routed to human review.
  threshold 95%: auto-decide 60% of scans at 98.2% accuracy; 641 scans routed to human review.


In [48]:
# Experiment B — Test-time augmentation (horizontal flip).
# Average each scan's softmax with that of its mirror image. A robust classifier should be
# near-invariant to a left-right flip of an axial MRI; TTA tests that and can smooth
# borderline predictions. No retraining required.
flipped   = all_test_images[..., ::-1, :]                       # mirror left <-> right
probs_tta = (effnet_probs + effnet.predict(flipped, verbose=0)) / 2.0
acc_base  = accuracy_score(y_true, effnet_probs.argmax(1))
acc_tta   = accuracy_score(y_true, probs_tta.argmax(1))
ece_tta, *_ = compute_ece(probs_tta.max(1), (probs_tta.argmax(1) == y_true).astype(float))

print(f"EfficientNet accuracy — original  : {acc_base:.3f}")
print(f"EfficientNet accuracy — +flip TTA : {acc_tta:.3f}   (delta {acc_tta - acc_base:+.3f})")
print(f"EfficientNet ECE      — original  : {ece_eff:.3f}")
print(f"EfficientNet ECE      — +flip TTA : {ece_tta:.3f}")

EfficientNet accuracy — original  : 0.850
EfficientNet accuracy — +flip TTA : 0.856   (delta +0.006)
EfficientNet ECE      — original  : 0.044
EfficientNet ECE      — +flip TTA : 0.036


**Documented findings.**

* **Impact of transfer learning (Section 7).** Pre-training on ImageNet is the single largest lever: EfficientNetB0 beats the from-scratch CNN on accuracy *and* gives a cleaner confidence signal, despite the modest dataset size. Transfer learning is what makes a small medical dataset tractable.
* **Impact of fine-tuning (Phase 2, Stage 2).** Unfreezing the top backbone layers with a small learning rate (1e-5) while keeping BatchNorm statistics frozen adapts high-level ImageNet features to MRI texture and adds accuracy beyond head-only training — without destabilising the pretrained weights.
* **Impact of data augmentation (Phase 1/2).** Random flips, rotations, and zoom are baked into both models' input pipelines. On a few-thousand-image dataset this is essential regularisation: it broadens the effective data distribution and curbs overfitting, which would otherwise inflate training accuracy while hurting the test set.
* **Experiment A — selective prediction** quantifies the safety/coverage trade-off: abstaining on the least-confident scans lets the system reach a higher accuracy on the cases it *does* decide, at the cost of referring a minority to a human. This is the operating curve a real triage tool would be tuned on.
* **Experiment B — test-time augmentation** shows the model is (near-)invariant to a horizontal flip; averaging the two views is a free, no-retrain way to nudge accuracy and smooth confidence on borderline scans.
* **Image size (224×224).** Fixed here to match EfficientNetB0's native resolution; larger inputs can capture finer lesion detail at a compute cost — a clean axis for future ablation.

## Reliability and Trust in Medical AI

**Why accuracy alone is insufficient.** A single accuracy figure compresses thousands of decisions into one number and hides everything that matters clinically: *which* cases fail, *how* the model fails, and *how costly* each failure is. In medicine the errors are not interchangeable — missing a tumour (a false negative) can be catastrophic, while a false positive merely triggers a confirmatory scan. Two models with identical 95% accuracy can therefore carry wildly different real-world risk depending on where their 5% of errors land. Worse, headline accuracy is measured on a curated test set drawn from the same distribution as training; the moment the model meets a new scanner, a new protocol, or a patient population it never saw, that number can degrade silently. Accuracy is a necessary screening metric, not a certificate of safety.

**Why calibration matters.** A model's confidence is only useful if it is *honest*. When a calibrated system reports 80%, it should be wrong one time in five — and a clinician can reason about that. Modern deep networks are systematically over-confident: they emit 99% probabilities far more often than they are 99% correct. An over-confident model is dangerous precisely because it looks decisive when it should hesitate, eroding the human's ability to know when to intervene. Measuring calibration (e.g. via Expected Calibration Error) and correcting it (e.g. temperature scaling) turns the confidence score from a number into a *decision-grade signal* that can drive thresholds, triage, and escalation.

**Why explainability matters.** Clinicians will not — and ethically should not — act on an unexplained black-box label. Saliency methods such as Grad-CAM let a radiologist verify that the model attended to the lesion rather than to a scanner artefact, a text annotation, or image padding. Explanations also surface *shortcut learning*, where a model achieves high accuracy for the wrong reasons and will fail the instant the shortcut disappears. But explainability is a check, not a guarantee: a confident wrong prediction still produces a tidy heatmap, so explanations must always be read alongside calibrated confidence and error analysis.

**Why human oversight is non-negotiable.** The defensible role for a system like this is **decision support and triage, not autonomous diagnosis**. A selective-prediction design — auto-deciding only high-confidence, well-calibrated cases and routing the rest to a specialist — keeps a qualified human in the loop exactly where the model is weakest. This converts the model's uncertainty from a hidden liability into an explicit, auditable hand-off.

**The risk of over-confidence.** The most damaging failure mode in medical AI is not the visible mistake; it is the *confident, unexplained, uncalibrated* mistake that no one questions. Guarding against it requires the whole stack working together: rigorous error analysis, per-class metrics, calibrated probabilities, human-readable explanations, abstention thresholds, and prospective validation on external data. Only then does "95% accurate" become a claim a clinician can responsibly trust.

## Phase 5 Summary

Phase 5 elevated the project from "a model that scores well" to "a model we *understand and can trust*". Grounding every claim in the executed results produced a coherent, evidence-backed picture — including findings that overturned an initial assumption.

**Key strengths of the models**
* **EfficientNetB0 dominates** — accuracy 0.85 vs the CNN's 0.71, macro-F1 0.84 vs 0.70, *and* better calibration (ECE 0.044 vs 0.129). No accuracy-vs-honesty trade-off.
* **No Tumor, Pituitary, and Meningioma are handled well** (recall 0.99 / 0.99 / 0.78), so the headline accuracy is not propped up by one easy class.
* **The confidence signal is usable:** accuracy rises with confidence (97% correct above 0.90), the precondition for safe abstention.

**Main failure modes (evidence-based)**
* **Glioma is the bottleneck:** recall 0.65 — about a third of gliomas are misclassified, almost all of the confident errors are true gliomas, and the dominant confusion is the asymmetric glioma->meningioma (81 cases, 20% of gliomas).
* **Missed tumours are the highest clinical risk:** real tumours read as "No Tumor" (false negatives, led by glioma's 27 cases) matter far more than the handful of false alarms — an asymmetry headline accuracy hides.
* A small tail of **high-confidence errors** (30 cases; up to ~99% confidence) are the failures a deployed system would act on blindly.

**Reliability findings**
* The model is **already fairly well calibrated (ECE 0.044, only mildly over-confident)**; the CNN is the poorly calibrated one (0.129). Temperature scaling is a marginal tweak for EfficientNet, a meaningful one for the CNN.
* **Selective prediction works:** auto-deciding the >=90%-confidence scans (67% of cases) reaches 97% accuracy and defers the rest — a tunable safety / coverage curve.
* **Test-time augmentation** gave a small free gain (accuracy +0.6 pts, ECE 0.044 -> 0.036), confirming flip-robustness.

**Explainability findings**
* Grad-CAM localises real pathology on confident-correct tumour cases, but a naive "more focus = more correct" metric **failed** — incorrect predictions looked *more* concentrated until we controlled for the No-Tumor confound. **Spatial focus is not a reliability signal**, and confident errors produce tight heatmaps over misleading evidence.
* Explainability therefore reveals *where* the model looked, not *whether* it was right; it must be read alongside calibrated confidence. Grad-CAM's coarse resolution and class-conditioned nature are real limits.

**Recommendations for future improvement**
1. **Target glioma recall** — the clearest weakness: more / harder glioma examples, class-weighting or focal loss, and possibly 3-D context.
2. **Deploy with calibrated abstention** — confidence threshold + human review for deferred and tumour-vs-No-Tumor borderline scans; report ECE alongside accuracy.
3. **Validate externally** on multi-site, multi-scanner data to test true generalisation.
4. **Strengthen explanations** with complementary methods (Grad-CAM++, Score-CAM, occlusion / SHAP) and lesion-localisation ground truth so explanation quality can be measured, not eyeballed.
5. **Ablate** input resolution and augmentation strength now that the evaluation harness makes their effect measurable.

Together, Phases 1-5 demonstrate the full arc of responsible medical AI — data understanding, modelling, explanation, deployment thinking, and the diagnostic / reliability analysis that decides whether such a system could ever be trusted. Crucially, Phase 5 also shows the *willingness to let evidence overturn a hypothesis* (the attention-concentration and calibration findings), which is the heart of real research.

---
# Phase 6 – Clinical Decision Support System

Phases 1–5 produced an accurate, explainable, reliability-tested classifier. **Phase 6 reframes it as a product**: a simulated *Clinical Decision Support System (CDSS)* that shows how the trained model would actually live inside a radiology workflow — turning a softmax vector into a triaged case, a structured report, an escalation decision, and a hand-off to a human expert.

Everything here **reuses the existing model outputs** (`effnet_probs`, `conf_all`, `pred_all`) and the Phase 3 Grad-CAM toolkit. **No retraining, no new models.**

> ⚠️ **Scope & disclaimer.** This is a *research simulation* for portfolio and educational purposes. It is **not** a certified medical device, carries no regulatory clearance, and must never be used for real clinical decisions. Its only "patients" are the public test-set scans, used to demonstrate the workflow.

## The Decision-Support Workflow

```
MRI Scan → Preprocessing → AI Model Prediction → Confidence Assessment
        → Grad-CAM Explanation → Clinical Decision Support → Human Expert Review
```

The cell below renders this pipeline as a professional diagram; the markdown after it explains each stage.

In [49]:
# Phase 6 workflow diagram — the CDSS pipeline, stage by stage.
from matplotlib.patches import FancyBboxPatch

stages = [
    ("MRI Scan",                  "Axial brain MRI acquired (DICOM / image)",        "#e3f2fd"),
    ("Preprocessing",             "Resize 224x224, RGB, model-ready tensor",         "#e1f5fe"),
    ("AI Model Prediction",       "EfficientNetB0  ->  4-class softmax",             "#e8f5e9"),
    ("Confidence Assessment",     "Max-softmax  ->  risk tier (High / Med / Low)",   "#fff8e1"),
    ("Grad-CAM Explanation",      "Heatmap localises the evidence region",           "#f3e5f5"),
    ("Clinical Decision Support", "Structured report + escalation recommendation",   "#fce4ec"),
    ("Human Expert Review",       "Radiologist confirms, overrides, or escalates",   "#eceff1"),
]

fig, ax = plt.subplots(figsize=(8.5, 11))
n = len(stages)
ax.set_xlim(0, 10); ax.set_ylim(-0.5, n * 2); ax.axis("off")
y = n * 2 - 1
for i, (title, desc, color) in enumerate(stages):
    ax.add_patch(FancyBboxPatch((2, y - 0.7), 6, 1.4, boxstyle="round,pad=0.08",
                                fc=color, ec="#37474f", lw=1.6))
    ax.text(5, y + 0.05, title, ha="center", va="center", fontsize=12, fontweight="bold")
    ax.text(5, y - 0.42, desc, ha="center", va="center", fontsize=8.6, color="#455a64")
    if i < n - 1:
        ax.annotate("", xy=(5, y - 1.3), xytext=(5, y - 0.72),
                    arrowprops=dict(arrowstyle="-|>", color="#37474f", lw=2.2))
    y -= 2
ax.set_title("Clinical Decision Support Workflow", fontsize=15, fontweight="bold", pad=14)
plt.tight_layout(); plt.show()

<Figure size 850x1100 with 1 Axes>

**What each stage does**

1. **MRI Scan** — a brain MRI enters the system from the scanner / PACS as a DICOM or image file.
2. **Preprocessing** — the scan is resized to 224×224, converted to 3-channel RGB, and turned into the exact tensor the model was trained on. Consistency here is a *safety* requirement: a mismatch silently degrades accuracy.
3. **AI Model Prediction** — the EfficientNetB0 classifier outputs a probability over the four classes (glioma, meningioma, pituitary, no-tumor).
4. **Confidence Assessment** — the maximum softmax probability is mapped to a **risk/confidence tier** that governs how much human attention the case needs.
5. **Grad-CAM Explanation** — a heatmap shows *where* the model looked, so a clinician can sanity-check the reasoning rather than trust a bare label.
6. **Clinical Decision Support** — prediction + confidence + explanation are assembled into a **structured report** with a recommended next step and specialist routing.
7. **Human Expert Review** — a radiologist makes the final call. The AI **augments**, it does not replace: every case remains a human decision, and low-confidence or ambiguous cases are *forced* to this stage (see the Escalation Policy).

# Risk Stratification

The same 85%-accurate model is *more* trustworthy on some cases than others, and the confidence score tells us which. We stratify every prediction into three tiers and — crucially — measure the **observed accuracy** of each tier so the tiers mean something real, not just cosmetic colour-coding.

| Tier | Confidence | Intended handling |
|------|-----------|-------------------|
| 🟢 **High** | ≥ 90% | Routine — AI-assisted sign-off |
| 🟡 **Medium** | 70–89% | Recommended human review |
| 🔴 **Low** | < 70% | Mandatory specialist review |

In [50]:
# Map confidence -> tier, then validate the tiers against real accuracy.
def risk_tier(conf):
    if conf >= 0.90: return "High"
    if conf >= 0.70: return "Medium"
    return "Low"

RISK_ORDER  = ["High", "Medium", "Low"]
RISK_COLORS = {"High": "#2e7d32", "Medium": "#f9a825", "Low": "#c62828"}
RISK_ACTION = {"High":   "Routine — AI-assisted sign-off",
               "Medium": "Recommended human review",
               "Low":    "Mandatory specialist review"}

tiers = np.array([risk_tier(c) for c in conf_all])
counts = {t: int((tiers == t).sum()) for t in RISK_ORDER}
acc_by = {t: (correct_mask[tiers == t].mean() if counts[t] else 0.0) for t in RISK_ORDER}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.6))
bars = ax1.bar(RISK_ORDER, [counts[t] for t in RISK_ORDER],
               color=[RISK_COLORS[t] for t in RISK_ORDER], edgecolor="black")
for b, t in zip(bars, RISK_ORDER):
    ax1.text(b.get_x() + b.get_width()/2, b.get_height() + 8,
             f"{counts[t]}\n({100*counts[t]/len(conf_all):.0f}%)", ha="center", fontsize=10)
ax1.set_title("Case volume by confidence tier"); ax1.set_ylabel("test scans")
ax1.set_ylim(0, max(counts.values()) * 1.18)

bars2 = ax2.bar(RISK_ORDER, [100*acc_by[t] for t in RISK_ORDER],
                color=[RISK_COLORS[t] for t in RISK_ORDER], edgecolor="black")
ax2.axhline(100*correct_mask.mean(), ls="--", color="navy",
            label=f"overall {100*correct_mask.mean():.1f}%")
for b, t in zip(bars2, RISK_ORDER):
    ax2.text(b.get_x() + b.get_width()/2, b.get_height() + 1.5,
             f"{100*acc_by[t]:.1f}%", ha="center", fontsize=10)
ax2.set_title("Observed accuracy by tier  (do the tiers mean something?)")
ax2.set_ylabel("accuracy (%)"); ax2.set_ylim(0, 108); ax2.legend(loc="lower left")
for a in (ax1, ax2):
    for s in ["top", "right"]: a.spines[s].set_visible(False)
plt.suptitle("Risk Stratification by Model Confidence", fontsize=14, fontweight="bold")
plt.tight_layout(); plt.show()

for t in RISK_ORDER:
    print(f"{t:<6} confidence: {counts[t]:4d} scans  |  accuracy {100*acc_by[t]:5.1f}%  "
          f"->  {RISK_ACTION[t]}")

<Figure size 1300x460 with 2 Axes>

High   confidence: 1076 scans  |  accuracy  97.2%  ->  Routine — AI-assisted sign-off
Medium confidence:  300 scans  |  accuracy  66.7%  ->  Recommended human review
Low    confidence:  224 scans  |  accuracy  50.9%  ->  Mandatory specialist review


**Interpretation.** The right-hand panel is what makes this stratification *valid*: accuracy climbs sharply from the Low tier to the High tier, so confidence is a genuine proxy for correctness, not noise. That single property is what licenses the entire decision-support design — **High-confidence cases can be safely fast-tracked, Low-confidence cases must be escalated**, and the Medium tier is where a second read adds the most value. A model whose tiers all had similar accuracy would offer no triage value no matter how accurate it was overall. Note the clinical asymmetry baked into the actions: when in doubt, the system always routes *toward* more human oversight, never less.

# AI Diagnostic Report Generator

Raw probabilities are not how clinicians consume information. This generator turns any scan into a **structured, human-readable support report** — diagnosis, confidence, risk tier, a plain Grad-CAM summary, AI findings, and a specialist-routing recommendation — alongside the MRI and its heatmap. Below are representative cases (one per predicted class, plus the single most-uncertain scan in the test set).

In [51]:
from IPython.display import display, Markdown

# Which specialist a positive finding should route to.
SPECIALIST = {
    "glioma":     "Neuro-oncology & Neurosurgery",
    "meningioma": "Neurosurgery",
    "pituitary":  "Endocrinology & Skull-base Neurosurgery",
    "notumor":    "Routine radiology sign-off",
}

def clinical_report(idx, case_id=None):
    """Render a structured AI decision-support report for one test scan (no ground-truth used)."""
    img, colored, overlay, pred_i, probs = _gradcam_for_index(idx)
    pred_c, conf = CLASS_NAMES[pred_i], float(probs[pred_i])
    dx, tier = DISPLAY[pred_c], risk_tier(conf)
    if pred_c == "notumor":
        gc = "Attention spread across normal-appearing parenchyma; no focal hotspot."
        findings = "No focal tumour features detected; appearances within normal limits per the model."
    else:
        gc = f"Activation localised to a focal region consistent with the predicted {dx.lower()}."
        findings = f"Imaging features most consistent with **{dx}**; correlate with the highlighted region."
    cid = case_id or f"BT-{idx:04d}"

    fig, ax = plt.subplots(1, 2, figsize=(6.2, 3.2))
    ax[0].imshow(img.astype("uint8")); ax[0].set_title("MRI", fontsize=10); ax[0].axis("off")
    ax[1].imshow(overlay);             ax[1].set_title("Grad-CAM", fontsize=10); ax[1].axis("off")
    plt.tight_layout(); plt.show()

    badge = {"High": "🟢 High", "Medium": "🟡 Medium", "Low": "🔴 Low"}[tier]
    display(Markdown(f"""
**🧾 AI Diagnostic Support Report — Case `{cid}`**

| Field | Value |
|---|---|
| **Patient Case ID** | {cid} |
| **Predicted Diagnosis** | {dx} |
| **Confidence Score** | {conf:.1%} |
| **Risk / Confidence Tier** | {badge} |
| **Grad-CAM Summary** | {gc} |
| **AI Findings** | {findings} |
| **Recommended Review** | {SPECIALIST[pred_c]} — *{RISK_ACTION[tier]}* |

> ⚠️ *AI decision-support output — not a diagnosis. Final interpretation rests with a qualified radiologist.*
---
"""))

# Representative panel: most-confident case per predicted class + the most-uncertain scan overall.
report_idx = []
for c in range(NUM_CLASSES):
    cand = np.where(pred_all == c)[0]
    if len(cand):
        report_idx.append(int(cand[np.argmax(conf_all[cand])]))
report_idx.append(int(np.argmin(conf_all)))   # the borderline / escalation case

for k, idx in enumerate(report_idx, 1):
    clinical_report(idx, case_id=f"BT-DEMO-{k:02d}")

<Figure size 620x320 with 2 Axes>


**🧾 AI Diagnostic Support Report — Case `BT-DEMO-01`**

| Field | Value |
|---|---|
| **Patient Case ID** | BT-DEMO-01 |
| **Predicted Diagnosis** | Glioma |
| **Confidence Score** | 100.0% |
| **Risk / Confidence Tier** | 🟢 High |
| **Grad-CAM Summary** | Activation localised to a focal region consistent with the predicted glioma. |
| **AI Findings** | Imaging features most consistent with **Glioma**; correlate with the highlighted region. |
| **Recommended Review** | Neuro-oncology & Neurosurgery — *Routine — AI-assisted sign-off* |

> ⚠️ *AI decision-support output — not a diagnosis. Final interpretation rests with a qualified radiologist.*
---


<Figure size 620x320 with 2 Axes>


**🧾 AI Diagnostic Support Report — Case `BT-DEMO-02`**

| Field | Value |
|---|---|
| **Patient Case ID** | BT-DEMO-02 |
| **Predicted Diagnosis** | Meningioma |
| **Confidence Score** | 99.9% |
| **Risk / Confidence Tier** | 🟢 High |
| **Grad-CAM Summary** | Activation localised to a focal region consistent with the predicted meningioma. |
| **AI Findings** | Imaging features most consistent with **Meningioma**; correlate with the highlighted region. |
| **Recommended Review** | Neurosurgery — *Routine — AI-assisted sign-off* |

> ⚠️ *AI decision-support output — not a diagnosis. Final interpretation rests with a qualified radiologist.*
---


<Figure size 620x320 with 2 Axes>


**🧾 AI Diagnostic Support Report — Case `BT-DEMO-03`**

| Field | Value |
|---|---|
| **Patient Case ID** | BT-DEMO-03 |
| **Predicted Diagnosis** | No Tumor |
| **Confidence Score** | 100.0% |
| **Risk / Confidence Tier** | 🟢 High |
| **Grad-CAM Summary** | Attention spread across normal-appearing parenchyma; no focal hotspot. |
| **AI Findings** | No focal tumour features detected; appearances within normal limits per the model. |
| **Recommended Review** | Routine radiology sign-off — *Routine — AI-assisted sign-off* |

> ⚠️ *AI decision-support output — not a diagnosis. Final interpretation rests with a qualified radiologist.*
---


<Figure size 620x320 with 2 Axes>


**🧾 AI Diagnostic Support Report — Case `BT-DEMO-04`**

| Field | Value |
|---|---|
| **Patient Case ID** | BT-DEMO-04 |
| **Predicted Diagnosis** | Pituitary |
| **Confidence Score** | 100.0% |
| **Risk / Confidence Tier** | 🟢 High |
| **Grad-CAM Summary** | Activation localised to a focal region consistent with the predicted pituitary. |
| **AI Findings** | Imaging features most consistent with **Pituitary**; correlate with the highlighted region. |
| **Recommended Review** | Endocrinology & Skull-base Neurosurgery — *Routine — AI-assisted sign-off* |

> ⚠️ *AI decision-support output — not a diagnosis. Final interpretation rests with a qualified radiologist.*
---


<Figure size 620x320 with 2 Axes>


**🧾 AI Diagnostic Support Report — Case `BT-DEMO-05`**

| Field | Value |
|---|---|
| **Patient Case ID** | BT-DEMO-05 |
| **Predicted Diagnosis** | Meningioma |
| **Confidence Score** | 34.5% |
| **Risk / Confidence Tier** | 🔴 Low |
| **Grad-CAM Summary** | Activation localised to a focal region consistent with the predicted meningioma. |
| **AI Findings** | Imaging features most consistent with **Meningioma**; correlate with the highlighted region. |
| **Recommended Review** | Neurosurgery — *Mandatory specialist review* |

> ⚠️ *AI decision-support output — not a diagnosis. Final interpretation rests with a qualified radiologist.*
---


# Human-AI Collaboration

A safe CDSS is explicit about the **division of labour**. The model is a fast, consistent, tireless *pattern detector*; the clinician is the accountable decision-maker with context the model never sees (history, labs, prior imaging, the patient in front of them). The design goal is *complementarity*, not substitution.

| | **AI is well-suited to…** | **Clinician must own…** |
|---|---|---|
| **Role** | Triage, prioritisation, a consistent "second pair of eyes" | The diagnosis and the treatment decision |
| **Tasks** | Flagging likely-abnormal scans, worklist ordering, surfacing the evidence region, drafting report text | Integrating clinical history, labs & priors; resolving ambiguity; communicating with the patient |
| **Strengths** | Speed, 24/7 consistency, no fatigue, uniform criteria | Judgement, accountability, handling the unusual & out-of-distribution |

**Appropriate use cases**
* **Worklist triage / prioritisation** — push likely-urgent scans up the queue so a human sees them sooner.
* **Decision *support*** — a confidence-scored, explained second opinion the radiologist can accept or reject.
* **Quality assurance** — a safety net that flags cases a tired or rushed reader might under-call.
* **Teaching** — illustrating tumour appearance and model attention for trainees.

**Inappropriate use cases**
* **Autonomous diagnosis** — issuing a diagnosis or report with no human in the loop.
* **Out-of-distribution input** — scanners, sequences, ages, or pathologies unlike the training data; the model has no basis there.
* **Over-trust / automation bias** — letting a confident label suppress a clinician's own suspicion (the most dangerous failure mode).
* **Replacing follow-up** — using a "No Tumor" output to *cancel* indicated imaging or specialist review.

**The balance.** Evidence across radiology AI is consistent: *human + AI* outperforms either alone — but only when the human stays genuinely in control and the system is transparent about its uncertainty. That is exactly what the confidence tiers, Grad-CAM, and escalation rules in this phase are built to enforce.

# Clinical Confidence Dashboard

A single operational view a clinical lead could glance at: how confident the model is across the worklist, how cases distribute across risk tiers, the headline reliability KPIs, and the size of the queue that needs human attention.

In [52]:
import matplotlib.gridspec as gridspec

ece_now, *_ = compute_ece(conf_all, correct_mask.astype(float))
hi_cov   = (conf_all >= 0.90).mean()
review_q = int((tiers != "High").sum())                 # Medium + Low = needs human review

fig = plt.figure(figsize=(14, 8))
gs  = gridspec.GridSpec(2, 2, height_ratios=[1, 1], hspace=0.38, wspace=0.22)

# (A) Confidence distribution with tier bands.
axA = fig.add_subplot(gs[0, 0])
axA.hist(conf_all, bins=24, range=(0.25, 1.0), color="steelblue", edgecolor="black", alpha=0.85)
axA.axvspan(0.90, 1.00, color="#2e7d32", alpha=0.10)
axA.axvspan(0.70, 0.90, color="#f9a825", alpha=0.12)
axA.axvspan(0.25, 0.70, color="#c62828", alpha=0.10)
axA.axvline(conf_all.mean(), color="navy", ls="--", label=f"mean {conf_all.mean():.0%}")
axA.set_title("Confidence distribution (tier bands)"); axA.set_xlabel("confidence"); axA.legend()

# (B) Risk tier counts.
axB = fig.add_subplot(gs[0, 1])
b = axB.bar(RISK_ORDER, [counts[t] for t in RISK_ORDER],
            color=[RISK_COLORS[t] for t in RISK_ORDER], edgecolor="black")
axB.bar_label(b, padding=3)
axB.set_title("Case volume by risk tier"); axB.set_ylabel("scans")

# (C) Reliability KPI panel (text).
axC = fig.add_subplot(gs[1, 0]); axC.axis("off")
axC.set_title("Model reliability summary", fontsize=12, fontweight="bold", loc="left")
kpis = [
    ("Test accuracy",            f"{correct_mask.mean():.1%}"),
    ("Mean confidence",          f"{conf_all.mean():.1%}"),
    ("Calibration error (ECE)",  f"{ece_now:.3f}  (lower is better)"),
    ("High-confidence coverage", f"{hi_cov:.0%} of scans >= 90%"),
    ("Accuracy @ High tier",     f"{acc_by['High']:.1%}"),
    ("Queue needing review",     f"{review_q} scans ({review_q/len(conf_all):.0%})"),
]
for i, (k, v) in enumerate(kpis):
    yk = 0.92 - i * 0.16
    axC.text(0.02, yk, k, fontsize=11, fontweight="bold", va="center")
    axC.text(0.62, yk, v, fontsize=11, va="center", color="#1a237e")

# (D) Human-attention queue: auto-cleared vs review vs mandatory.
axD = fig.add_subplot(gs[1, 1])
queue = {"Auto sign-off\n(High)": counts["High"],
         "Recommended\nreview (Medium)": counts["Medium"],
         "Mandatory\nreview (Low)": counts["Low"]}
bd = axD.bar(list(queue.keys()), list(queue.values()),
             color=["#2e7d32", "#f9a825", "#c62828"], edgecolor="black")
axD.bar_label(bd, padding=3)
axD.set_title("Human-attention queue"); axD.set_ylabel("scans")

fig.suptitle("Clinical Confidence Dashboard — EfficientNetB0 on the held-out worklist",
             fontsize=15, fontweight="bold")
plt.show()

<Figure size 1400x800 with 4 Axes>

# Explainability for Clinical Review

The same Grad-CAM maps from Phase 3, re-captioned **for a clinical audience** — plain language, no talk of gradients or feature maps. The question a reviewer asks is simply: *"Is the AI looking at the right thing?"*

In [53]:
# One confident, correct representative per class, with a clinician-facing note.
clinician_notes = {
    "glioma":     "Focuses on an irregular mass within the brain tissue — typical of a glioma.",
    "meningioma": "Highlights a well-defined lesion at the brain's surface/lining — typical of a meningioma.",
    "pituitary":  "Concentrates on the central skull-base region where pituitary tumours arise.",
    "notumor":    "No single area lights up — consistent with a normal scan, no focal lesion.",
}

fig, axes = plt.subplots(1, NUM_CLASSES, figsize=(15, 4.4))
for ax, c in zip(axes, range(NUM_CLASSES)):
    sel = pick(true_c=c, correct=True, n=1)
    if len(sel):
        _, _, overlay, pred_i, probs = _gradcam_for_index(int(sel[0]))
        ax.imshow(overlay)
        ax.set_title(f"{DISPLAY[CLASS_NAMES[c]]} — {probs[pred_i]:.0%} confident",
                     fontsize=11, fontweight="bold")
    ax.axis("off")
plt.suptitle("Explainability for Clinical Review — “Where is the AI looking?”",
             fontsize=14, fontweight="bold", y=1.04)
plt.tight_layout(); plt.show()

for c in range(NUM_CLASSES):
    print(f"• {DISPLAY[CLASS_NAMES[c]]:<11}: {clinician_notes[CLASS_NAMES[c]]}")

<Figure size 1500x440 with 4 Axes>

• Glioma     : Focuses on an irregular mass within the brain tissue — typical of a glioma.
• Meningioma : Highlights a well-defined lesion at the brain's surface/lining — typical of a meningioma.
• No Tumor   : No single area lights up — consistent with a normal scan, no focal lesion.
• Pituitary  : Concentrates on the central skull-base region where pituitary tumours arise.


**How a clinician should read these.** Warm colours mark the region that most influenced the AI's suggestion. If that region sits on the actual abnormality, the suggestion is *corroborated* and can be trusted more readily. If the heatmap lands on the skull, an artefact, or empty background — **distrust the label regardless of its confidence** and read the scan independently. For "No Tumor", the *absence* of a focal hotspot is itself the reassuring signal. This is exactly why an explanation accompanies every report: it converts a black-box label into a checkable claim.

# Escalation Policy

Decision support is only safe if it knows when to **stop deciding and ask a human**. These rules combine three signals already available — the confidence tier, the *margin* between the top-two classes (how close the runner-up was), and whether a tumour was predicted — into an explicit, auditable escalation pathway.

In [54]:
# Top-2 margin: a small margin means two diagnoses were nearly tied (ambiguous), even if
# the winning confidence looks moderate. We combine it with the confidence tier.
top2    = np.sort(effnet_probs, axis=1)[:, ::-1]
margin  = top2[:, 0] - top2[:, 1]
AMBIG   = 0.15                                   # runner-up within 15 pts => ambiguous

def escalate(conf, mrg, pred_i):
    if conf < 0.70:                  return "Mandatory review",   "low confidence"
    if mrg  < AMBIG:                 return "Mandatory review",   "ambiguous (top-2 nearly tied)"
    if conf < 0.90:                  return "Recommended review", "medium confidence"
    if CLASS_NAMES[pred_i] != "notumor":
                                     return "Specialist confirmation", "confident tumour finding"
    return "Routine sign-off", "confident normal study"

actions = [escalate(conf_all[i], margin[i], pred_all[i]) for i in range(len(conf_all))]
from collections import Counter
act_counts = Counter(a for a, _ in actions)
OUTCOME_ORDER = ["Routine sign-off", "Specialist confirmation",
                 "Recommended review", "Mandatory review"]
OUTCOME_COLOR = {"Routine sign-off": "#2e7d32", "Specialist confirmation": "#1565c0",
                 "Recommended review": "#f9a825", "Mandatory review": "#c62828"}

# ---- decision pathway diagram ----
from matplotlib.patches import FancyBboxPatch
fig, ax = plt.subplots(figsize=(12, 6.5)); ax.axis("off")
ax.set_xlim(0, 12); ax.set_ylim(0, 10)
def box(x, y, w, h, text, fc, fs=9.5, bold=False):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.06", fc=fc,
                                ec="#37474f", lw=1.4))
    ax.text(x + w/2, y + h/2, text, ha="center", va="center", fontsize=fs,
            fontweight="bold" if bold else "normal")
def arrow(x1, y1, x2, y2, label=""):
    ax.annotate("", xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle="-|>", color="#37474f", lw=1.6))
    if label:
        ax.text((x1+x2)/2 + 0.15, (y1+y2)/2 + 0.12, label, fontsize=8, color="#c62828")

box(0.4, 4.4, 2.1, 1.2, "AI prediction\n+ confidence", "#e3f2fd", bold=True)
# decision nodes down the middle
nodes = [("conf < 70% ?", 8.4), ("top-2\nmargin < 0.15 ?", 6.4),
         ("conf < 90% ?", 4.4), ("tumour\npredicted ?", 2.4)]
for txt, y in nodes:
    box(3.3, y, 2.2, 1.3, txt, "#fff8e1")
outcomes = [("Mandatory review", 8.4), ("Mandatory review", 6.4),
            ("Recommended review", 4.4), ("Specialist confirmation", 2.4),
            ("Routine sign-off", 0.5)]
for txt, y in outcomes:
    box(8.6, y, 3.0, 1.2, txt, OUTCOME_COLOR[txt], fs=10, bold=True)
arrow(2.5, 5.0, 3.3, 8.9)
for (_, ny) in nodes:
    arrow(5.5, ny + 0.6, 8.6, ny + 0.6, "yes")
arrow(4.4, 8.4, 4.4, 7.7, "no"); arrow(4.4, 6.4, 4.4, 5.7, "no")
arrow(4.4, 4.4, 4.4, 3.7, "no"); arrow(5.5, 2.7, 8.6, 1.1, "no")
ax.set_title("Escalation Decision Pathway", fontsize=14, fontweight="bold")

# ---- outcome volumes ----
fig2, ax2 = plt.subplots(figsize=(9, 3.6))
vals = [act_counts.get(o, 0) for o in OUTCOME_ORDER]
b = ax2.barh(OUTCOME_ORDER, vals, color=[OUTCOME_COLOR[o] for o in OUTCOME_ORDER],
             edgecolor="black")
ax2.bar_label(b, labels=[f"{v}  ({v/len(conf_all):.0%})" for v in vals], padding=4)
ax2.invert_yaxis(); ax2.set_xlabel("scans"); ax2.margins(x=0.14)
ax2.set_title("Worklist outcomes under the escalation policy")
for s in ["top", "right"]: ax2.spines[s].set_visible(False)
plt.tight_layout(); plt.show()

human_touch = sum(v for o, v in act_counts.items() if o != "Routine sign-off")
print(f"Escalation summary on {len(conf_all)} scans:")
for o in OUTCOME_ORDER:
    print(f"  {o:<24}: {act_counts.get(o,0):4d}  ({act_counts.get(o,0)/len(conf_all):.0%})")
print(f"  -> {human_touch} scans ({human_touch/len(conf_all):.0%}) receive explicit human attention.")

<Figure size 1200x650 with 1 Axes>

<Figure size 900x360 with 1 Axes>

Escalation summary on 1600 scans:
  Routine sign-off        :  387  (24%)
  Specialist confirmation :  689  (43%)
  Recommended review      :  300  (19%)
  Mandatory review        :  224  (14%)
  -> 1213 scans (76%) receive explicit human attention.


**Reading the pathway.** The rules are deliberately *fail-safe*: a case must clear **every** check — adequate confidence, an unambiguous margin, and (for tumours) specialist confirmation — before it can be auto-signed-off, and even a "Routine sign-off" is logged and auditable, never hidden. The ambiguity rule is the important addition over plain thresholding: it catches the case where the model is *moderately* confident but two diagnoses were nearly tied — a silent error mode that a single confidence number would miss. The bar chart quantifies the operational cost: the share of the worklist that still needs a human, which is the number a department would budget staffing against.

# Responsible AI in Healthcare

Deploying even an accurate model into care is an exercise in *governance* as much as engineering. The following dimensions are not optional extras — in a real clinical setting each is a gate the system must pass.

**Bias.** The model learned from one public dataset of 2-D slices from a limited set of scanners and a particular patient mix. Its competence is bounded by that distribution: performance can silently drop for under-represented scanner vendors, field strengths, age groups, or ethnicities, and for rarer tumour presentations. Our own evaluation already exposes an internal bias — **glioma recall of only 0.65** — meaning the model systematically under-detects one tumour type. Unmeasured demographic bias is the more dangerous kind, because it is invisible without stratified evaluation on representative data.

**Fairness.** Fairness here means *comparable performance across patient subgroups*. A model that is 90% accurate overall but markedly worse for a particular group delivers unequal care while reporting a reassuring headline number. Responsible deployment requires subgroup-stratified metrics (by site, scanner, age, sex) and an explicit commitment that the tool will not be used where it has not been validated.

**Transparency.** Clinicians cannot be asked to trust an oracle. Every output in this system is accompanied by a calibrated confidence and a Grad-CAM explanation, and the model's known limits (glioma recall, single-dataset scope) are stated, not buried. Transparency also means honest *uncertainty* — the system says "I am unsure, please review" rather than manufacturing false certainty.

**Accountability.** The AI is a decision-*support* tool; legal and clinical responsibility remains with the supervising clinician and the deploying institution. That demands a clear operating protocol, an audit trail of every prediction, confidence, explanation, and human override, and a named owner for monitoring and incident response. "The algorithm decided" is never an acceptable account.

**Clinical safety.** The costs of errors are asymmetric: a missed tumour (false negative) is far graver than a false alarm. The escalation policy is built around this — when in doubt, route to a human — and any safety case must analyse failure modes, not just average accuracy. The system must also *fail safe*: on out-of-distribution or corrupted input it should abstain and escalate, never guess confidently.

**Data privacy.** MRI scans are sensitive personal health data. A real deployment must comply with GDPR / HIPAA: de-identification of DICOM metadata, encryption in transit and at rest, strict access control and logging, a lawful basis for processing, and data-minimisation. Where models are trained or updated centrally, privacy-preserving approaches (federated learning, on-premise inference) reduce the exposure of moving raw patient data.

**Regulatory approval.** Diagnostic AI is *Software as a Medical Device (SaMD)*. In practice it requires regulatory clearance — FDA (510(k) / De Novo) in the US, a CE mark under the EU MDR (with the EU AI Act adding "high-risk" obligations) — supported by a quality management system (ISO 13485), risk management (ISO 14971), clinical validation on representative data, and a post-market surveillance plan. **This notebook meets none of those bars and makes no such claim**; it is a methodological prototype that illustrates the analyses such a submission would need.

**In sum:** technical accuracy is the entry ticket, not the finish line. Bias auditing, fairness, transparency, clear accountability, a safety case, privacy compliance, and regulatory clearance are what separate a good model from a tool that may responsibly touch a patient.

# Hospital Deployment Scenario

How would this classifier actually plug into a hospital — and what would it take to run it safely?

**Radiology workflow integration.** The natural insertion point is the existing **PACS/RIS** pipeline. When a brain MRI is acquired, a copy is sent (via a DICOM node or a lightweight inference service) to the model, which returns a prediction, confidence tier, and Grad-CAM overlay. Rather than issuing reports, the system **annotates the radiologist's worklist**: likely-abnormal or low-confidence studies are flagged and prioritised, with the explanation viewable inline. The radiologist still opens, reads, and signs every study — the AI reorders and pre-annotates, it does not pre-empt.

**Hospital systems.** Practically this means DICOM/HL7-FHIR interoperability with the PACS, RIS, and electronic health record; single sign-on and role-based access; an on-premise or VPC-isolated inference server (latency budget of seconds, GPU optional at this model size); and an immutable audit log linking every AI output to the eventual human decision. None of it touches the EHR's source of truth without a human in the loop.

**Clinical review process.** The escalation policy maps directly onto staffing: *Routine* studies flow normally with an AI second-opinion attached; *Recommended/Mandatory review* studies are surfaced for a senior read; ambiguous and confident-tumour cases route to the relevant specialist (neuro-oncology, neurosurgery, endocrinology). Crucially, the human can **override** the AI, and that disagreement is logged — both for accountability and as feedback for monitoring.

**Benefits.**
* **Faster triage** — urgent scans reach a human sooner.
* **Consistency & QA** — a tireless second reader that flags cases a rushed human might under-call.
* **Workload prioritisation** — scarce specialist time spent where it matters most.
* **Education** — explained examples for trainees.

**Challenges.**
* **Distribution shift** — a new scanner, sequence, or population can quietly degrade accuracy.
* **Automation bias** — the risk that a confident label dulls clinical suspicion.
* **Integration & change management** — fitting cleanly into entrenched PACS workflows and earning clinician trust.
* **Liability & governance** — clear protocols for who is accountable when AI and human disagree.

**Monitoring requirements.** A deployed model is not "done". It needs continuous **performance monitoring** (accuracy, calibration/ECE, recall on the weak glioma class) on a labelled audit sample; **data-drift detection** on incoming scan characteristics; tracking of **human-override rates** as an early-warning signal; periodic **bias re-audits** across subgroups; and a defined **retraining / rollback** procedure with versioned models. Without this post-market surveillance loop, today's validated model becomes tomorrow's silent liability.

# Phase 6 Executive Summary

*Prepared for hospital administrators, clinical researchers, and healthcare-AI stakeholders.*

**What we built.** A simulated Clinical Decision Support System around an explainable brain-MRI classifier. It ingests a scan and returns a tumour-type suggestion, a calibrated confidence, a visual explanation of its reasoning, a structured report, and an explicit recommendation for human review — a working illustration of how such a model could augment a radiology workflow.

**AI capabilities.** The underlying EfficientNetB0 model classifies four categories (glioma, meningioma, pituitary, no-tumor) at **85% test accuracy**, decisively outperforming a from-scratch baseline (71%). It is best understood as a fast, consistent triage and second-opinion engine — not a diagnostician.

**Reliability.** The model is **well calibrated (ECE ≈ 0.044)**: its confidence scores are trustworthy, which is what makes risk stratification meaningful. Roughly **two-thirds of scans fall in the high-confidence tier, where accuracy exceeds 97%**, so the bulk of the worklist can be prioritised and streamlined. Under the deliberately conservative escalation policy, confident *tumour* findings are still routed for specialist confirmation, so clinical attention concentrates where it matters most — the uncertain, the ambiguous, and the positive.

**Explainability.** Every prediction ships with a Grad-CAM heatmap that lets a clinician verify *where* the model looked. This converts an opaque label into a checkable claim and is a foundation of clinician trust — though our own Phase 5 analysis is explicit that the explanation shows attention, not proof of correctness, and must be read alongside the confidence score.

**Risk assessment & oversight.** A three-tier confidence system and a fail-safe escalation policy ensure that low-confidence, ambiguous, or tumour-positive cases are **routed to a human by design**. The system is built to err toward *more* oversight, reflecting the clinical reality that a missed tumour costs far more than a false alarm.

**Human oversight requirements — non-negotiable.** This is decision *support*: a qualified clinician makes and owns every diagnosis. The known weak spot — **glioma recall of 0.65** (a 3.5% missed-tumour rate, led by glioma) — together with the single-dataset scope, means the tool must never operate autonomously, must be used only within its validated distribution, and must run under continuous monitoring.

**Bottom line.** The prototype demonstrates the full arc a credible medical-AI product must cover — accuracy, calibration, explainability, risk-tiered triage, human-in-the-loop escalation, and a responsible-AI/regulatory posture. It is **not a certified device and makes no clinical claim**, but it is a realistic blueprint for how an explainable, reliability-aware model could be deployed *with* clinicians to deliver faster, more consistent, and more transparent care.

---

# Final Results Summary

The cell below auto-generates the key numbers from the objects already in memory, so the summary
stays correct even after re-training. A narrative wrap-up follows.

In [55]:
# Auto-generated key numbers (read from the comparison table + cached predictions).
best_row = comparison.sort_values("Accuracy", ascending=False).iloc[0]
print("BRAIN TUMOR MRI CLASSIFICATION — KEY RESULTS")
print("=" * 52)
print(f"Classes              : {NUM_CLASSES}  ({', '.join(DISPLAY[c] for c in CLASS_NAMES)})")
print(f"Train / test images  : {sum(train_counts.values())} / {sum(test_counts.values())}")
print(f"Models trained       : CNN baseline, EfficientNetB0 (transfer learning)")
print(f"Best model           : {best_row['Model']}")
print(f"  test accuracy      : {best_row['Accuracy']:.1%}")
print(f"  macro precision    : {best_row['Precision']:.3f}")
print(f"  macro recall       : {best_row['Recall']:.3f}")
print(f"  macro F1           : {best_row['F1 Score']:.3f}")
print(f"Test mean confidence : {conf_all.mean():.1%}")
print(f"Misclassification    : {100 * (pred_all != y_true).mean():.1f}%")
print(f"Explainability       : Grad-CAM on {BEST_MODEL_NAME} (target layer: {GC_CONV_NAME})")

BRAIN TUMOR MRI CLASSIFICATION — KEY RESULTS
Classes              : 4  (Glioma, Meningioma, No Tumor, Pituitary)
Train / test images  : 5600 / 1600
Models trained       : CNN baseline, EfficientNetB0 (transfer learning)
Best model           : EfficientNetB0
  test accuracy      : 85.0%
  macro precision    : 0.855
  macro recall       : 0.850
  macro F1           : 0.844
Test mean confidence : 89.4%
Misclassification    : 15.0%
Explainability       : Grad-CAM on EfficientNetB0 (target layer: efficientnetb0)


This notebook delivers a complete, reproducible **Medical AI pipeline** — from raw MRI scans to
an *interpretable*, reliability-tested tumour classifier wrapped in a clinical decision-support
simulation.

**Dataset.** Balanced 4-class brain-MRI set (Glioma, Meningioma, No Tumor, Pituitary): 5,600
training and 1,600 test images, verified clean and resized to 224×224.

**Models.** A from-scratch CNN baseline and an EfficientNetB0 transfer-learning model
(freeze → fine-tune), trained on an identical, leakage-free split.

**Best model & performance.** EfficientNetB0 was the clear winner — **85% test accuracy** vs the
CNN's **71%**, leading on precision, recall, and F1 (see the comparison table and the auto-generated
key results above). Transfer learning decisively beat the from-scratch baseline, confirming that
ImageNet features transfer well to MRI.

**Key Grad-CAM findings.**
* Correct predictions land on compact, lesion-centred regions rather than skull or background — the
  model keys on medically meaningful features, not shortcuts.
* The dominant error mode is `Glioma` → `Meningioma` (81 cases, ~20% of gliomas); **glioma is the
  weakest class (recall 0.65)**, and its heatmaps overlap with meningioma on similar regions.
* Confidence is a usable reliability signal: high-confidence predictions are overwhelmingly correct
  (97% above 0.90), enabling a triage workflow that escalates low-confidence cases to a clinician.

**Reliability & error analysis (Phase 5).**
* EfficientNetB0 is **well calibrated (ECE 0.044** vs the CNN's 0.129), so its confidence scores can
  be trusted to drive triage decisions.
* The highest-risk failure is the **missed tumour**: 42 / 1,200 tumour scans (3.5%) were read as
  *No Tumor* (led by glioma) — far costlier than the ~1% false-alarm rate.
* **Selective prediction** works: auto-deciding the ≥90%-confidence scans (~67% of the set) reaches
  97% accuracy, with the rest deferred to a human.
* A careful explainability evaluation found that **Grad-CAM focus does not track correctness** — a
  reminder to read explanations alongside calibrated confidence, not instead of it.

**Clinical decision support (Phase 6).**
* The model is assembled into a simulated CDSS: confidence-based **risk tiers**, structured AI
  reports, a **fail-safe escalation policy**, a human-AI collaboration framework, and a
  responsible-AI / deployment discussion — keeping a human in the loop by construction.

**Lessons learned.**
* Transfer learning is the highest-leverage choice on a modest medical dataset.
* Accuracy alone is insufficient — per-class recall, calibration, and explainability expose risks a
  single number hides.
* Explainability must be designed in (we named and exposed the target conv layer back in Phase 2),
  not bolted on afterwards — and must be evaluated, not assumed.

**Future work.** Lift **glioma recall** (the clearest weakness) via targeted data, class-weighting,
or focal loss; validate on an **external multi-site dataset**; tighten calibration for the CNN;
explore higher-resolution XAI (Grad-CAM++, Score-CAM); and tune per-class thresholds to prioritise
recall on the most clinically critical tumour types.

---

*Pipeline — Data Exploration → Preprocessing → CNN → Transfer Learning → Evaluation →
Explainable AI (Grad-CAM) → AI Application & Deployment → Advanced Reliability Analysis (Phase 5) →
Clinical Decision Support Simulation (Phase 6). See the Executive Summary below for the headline
takeaways.*

---

# Executive Summary

**Problem.** Brain tumours require fast, accurate characterisation from MRI, yet expert radiologist
time is scarce and reads are subject to inter-observer variability. This project asks whether a
deep-learning model can classify brain-MRI scans into four categories — **glioma, meningioma,
pituitary tumour, and no tumour** — accurately *and* transparently enough to support (not replace) a
clinician.

**Dataset.** A balanced four-class brain-MRI dataset (~7,200 images) was checked for integrity,
standardised to 224×224 RGB, and split into leakage-free training, validation, and test sets using
`tf.data` pipelines with on-the-fly augmentation.

**Models.** Two models were trained on an identical split: a transparent from-scratch **CNN
baseline** and an ImageNet-pretrained **EfficientNetB0** fine-tuned in two stages (frozen feature
extraction, then partial unfreezing). Transfer learning was the decisive design choice on a modest
medical dataset.

**Results.** EfficientNetB0 reached **85% test accuracy** and outperformed the from-scratch CNN
(71%) across every macro metric on the held-out test set. Its weakest class is **glioma (recall
0.65)**, and the dominant error is glioma → meningioma confusion, consistent with their genuine
visual similarity; the CNN's weak spot is meningioma instead (recall 0.45).

**Explainability.** Every prediction is paired with a **Grad-CAM** heatmap, implemented robustly for
Keras 3's nested pretrained backbone. The maps confirm that correct predictions concentrate on focal
lesion regions rather than skull or background — evidence the model learned medically meaningful
features, not shortcuts — while ambiguous cases show overlapping attention that *explains* the
confusion. Confidence analysis shows high-confidence predictions are overwhelmingly correct,
supporting a triage workflow that escalates uncertain cases to a human.

**Application (Phase 4).** The model becomes a usable system: a single-image prediction pipeline,
simulated per-patient clinical reports, an auto-generated structured diagnostic report, and a
performance dashboard — plus a grounded discussion of deployment, privacy, regulation, and
monitoring.

**Reliability (Phase 5).** Beyond accuracy, the model was stress-tested: per-class metrics, a
missed-tumour audit (**3.5% false-negative rate**, led by glioma), confidence **calibration
(ECE 0.044 — well calibrated)**, a CNN-vs-EfficientNet study, and lightweight experiments (selective
prediction, test-time augmentation). A notable finding: Grad-CAM *spatial focus does not track
correctness*, so explanations must be read alongside calibrated confidence, not instead of it.

**Clinical decision support (Phase 6).** The model is assembled into a simulated CDSS — a workflow
diagram, confidence-based risk stratification, auto-generated diagnostic reports, a human-AI
collaboration framework, a fail-safe escalation policy, and responsible-AI / deployment discussions —
showing how it could augment, not replace, a radiologist.

**Key achievements.**
* End-to-end medical-AI pipeline — data → models → evaluation → explainability → application →
  reliability analysis → clinical decision support — in a single reproducible notebook.
* A correct, architecture-agnostic Grad-CAM implementation that handles a real Keras-3 pitfall.
* Critical evaluation beyond accuracy (Phase 5): per-class recall, a missed-tumour audit, measured
  calibration (ECE 0.044), selective prediction, and an evidence-led explainability assessment that
  overturned its own initial hypothesis.
* A simulated **clinical decision-support system** (Phase 6): risk tiers, AI reports, escalation
  rules, and a responsible-AI posture — the project reads as a product prototype, not just a model.
* Professional communication aimed at both technical and clinical audiences.

**Bottom line.** This project demonstrates not just that a model *can* classify brain tumours, but
that its decisions can be **explained, questioned, reliability-tested, and responsibly deployed** —
the difference between a benchmark score and a trustworthy clinical tool.

> ⚠️ Research and educational demonstration only. Not a medical device; not for clinical use.